# dialog

> dialog provides message and dialog introspection, and import/export utilities for solveit dialogs.

In [ ]:
#| default_exp dialog

In [ ]:
#| export
#| hide
import re, json, ast, warnings, copy, asttokens
from html import escape
from pathlib import Path
from typing import TypeAlias
from IPython import get_ipython
from mistletoe import Document
from mistletoe.block_token import CodeFence
from mistletoe.markdown_renderer import MarkdownRenderer
from nbdev.process import extract_directives
from fastcore.foundation import patch, adict, AttrDict as AD
from fastcore.meta import delegates
from fastcore.py2pyi import functypes
from dialoghelper.core import add_msg, read_msg, find_msgs, msg_idx, run_msg, toggle_header, find_dname
from dialoghelper.utils import ast_py
from pote.dutil import in_dialog

In [ ]:
#| hide
from IPython.display import HTML, Markdown
from nbformat.v4 import new_notebook, new_code_cell, writes_json
import fastcore.all as FC
from fastcore.test import *
from fastcore.py2pyi import _proc_patches, has_deco, _body_ellip
from nbdev.config import get_config, is_nbdev
from fasthtml.components import Div, Pre
from mistletoe import HtmlRenderer 
from pote.dutil import link_msg

## message utils
Core primitives for working with solveit messages: a `Msg` type alias, utilities for reading nbdev directives and message metadata, progressive disclosure of note content, and an extended `MsgX` subclass with convenience properties.

### message tag

Defines `Msg` as a type alias for `adict`, providing a concise, readable type annotation for solveit message objects throughout the dialog.

In [ ]:
#| export
Msg: TypeAlias = AD

### message metadata & directives

Utilities for inspecting solveit message metadata. `is_exported` checks if a message is marked for export (via solveit metadata or nbdev `#|export`/`#|exports`/`#|exporti` directives). `is_hidden` checks if a message is hidden from doc generation via `#|hide`. `is_echo` checks if a code message is marked to show source via `#|echo`. `is_header` checks if the note message start with a markdown heading.

In [ ]:
cell = AD(source='#|exporti\nprint(13)')
dirs = extract_directives(cell, remove=False)
bool(dirs.keys() & {'exporti', 'exports', 'export'})

True

In [ ]:
print(extract_directives(AD(source='#| echo\nprint(13)'), remove=False))
print(extract_directives(AD(source='#| echo: true\nprint(13)'), remove=False))

{'echo': []}
{'echo:': ['true']}


In [ ]:
#| exporti
_exports = {'exporti', 'exports', 'export'}
_hidden = {'hide'}
_echo = {'echo', 'echo:'}

In [ ]:
#| export
def is_exported(m:Msg) -> bool:
    "Is message exported with solveit metadata or nbdev directive?"
    return m.is_exported or (m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _exports))

def is_hidden(m:Msg) -> bool:
    "Is message hidden by a nbdev directive (only for doc generation, not AI visibility)?"
    return m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _hidden)

def is_echo(m:Msg) -> bool:
    "Does message have an echo directive that isn't false?"
    if m.msg_type != 'code': return False
    d = {k:v for k,v in extract_directives(AD(source=m.content), remove=False).items() if k in _echo}
    return bool(d) and all('false' not in v for v in d.values())

def is_header(m:Msg) -> bool:
    "Does note message content start by a markdown header?"
    return m.msg_type == 'note' and m.content.startswith('#')

def is_skipped(m:Msg) -> bool: return bool(m.skipped)

In [ ]:
test_is(is_exported(await read_msg()) and is_exported(await read_msg(-2)), True)
test_is(is_exported(await read_msg(-4)), False)

In [ ]:
#| echo
test_is(is_hidden(Msg(msg_type='code', content='#|hide\n\nsome content')), True)

test_is(is_header(Msg(msg_type='note', content='# a')), True)
test_is(is_header(Msg(msg_type='note', content=' ### a')), False)

test_is(is_echo(Msg(msg_type='code', content='#| echo\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#|hide\n#| echo: true\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#| echo: false\nprint(1)')), False)
test_is(is_echo(Msg(msg_type='note', content='#| echo\nsome text')), False)
test_is(is_echo(Msg(msg_type='code', content='print(1)')), False)

### notes disclosure

`clamp_note` supports four levels of **progressive disclosure** for markdown notes.
| Level | Returns |
|---|---|
| `'header'` | The first line, if it is a markdown header; otherwise `''` |
| `'header+'` | Header + first non-blank line after it |
| `'header++'` | Header + first paragraph, only if it immediately follows the header |
| `int` (e.g. `30`) | First *n* lines, regardless of structure |
All levels return `''` if the content is empty or (for header levels) has no leading header.

In [ ]:
_section = (await read_msg()).content

In [ ]:
#| export
def clamp_note(
    c:str, # Markdown note content
    level:str|int='header' # Disclosure level: 'header', 'header+', 'header++', or int for max lines
) -> str:
    "Return a progressively disclosed excerpt of a markdown note."
    c = c.strip().splitlines()
    if c:
        if isinstance(level, int): return '\n'.join(c[:level])
        if level == 'header': return c[0] if c[0][0] == '#' else ''
        if level == 'header+':
            i, res = 0, c[0]
            if res[0] != '#': return ''
            while (i:=i+1) < len(c):
                if c[i]: return f"{res}\n{c[i]}"
            return res
        if level.startswith('header++'):
            header = c[0] if c[0][0] == '#' else ''
            if not header: return ''
            blocks = [b.strip() for b in '\n'.join(c[1:]).split('\n\n') if b.strip()]
            if blocks:
                first = blocks[0].splitlines()[0]
                if not first.startswith('#') and not first.startswith(('```', '~~~')):
                    return f"{header}\n{blocks[0]}"
            return header
    return ''

In [ ]:
# Paragraph immediately follows header
test_eq(clamp_note('# hdr\n\nParagraph text.', 'header++'), '# hdr\nParagraph text.')

# Multi-line paragraph immediately follows header
test_eq(clamp_note('# hdr\n\nLine one.\nLine two.', 'header++'), '# hdr\nLine one.\nLine two.')

# First block is a subheader — return header only
test_eq(clamp_note('# hdr\n\n## subheader\n\nActual paragraph.', 'header++'), '# hdr')

# First block is a code fence — return header only
test_eq(clamp_note('# hdr\n\n```python\ncode\n```\n\nReal paragraph.', 'header++'), '# hdr')

# No content after header
test_eq(clamp_note('# hdr', 'header++'), '# hdr')

# No header
test_eq(clamp_note('just text\n\nParagraph.', 'header++'), '')

# Lines
test_eq(clamp_note('just text\n\nParagraph.', 30), 'just text\n\nParagraph.')
test_eq(clamp_note('a\n'*50, 30), ('a\n'*30)[:-1])

# Empty / whitespace
test_eq(clamp_note('', 'header++'), '')
test_eq(clamp_note('   ', 'header++'), '')

In [ ]:
c = _section
test_eq(clamp_note(c), '### notes disclosure')
test_eq(clamp_note(c, 'header+').splitlines()[1][:21], '`clamp_note` supports')
test_eq(clamp_note(c, 5).splitlines()[:5], c.splitlines()[:5])
print(clamp_note(c, 'header++'))

### notes disclosure
`clamp_note` supports four levels of **progressive disclosure** for markdown notes.
| Level | Returns |
|---|---|
| `'header'` | The first line, if it is a markdown header; otherwise `''` |
| `'header+'` | Header + first non-blank line after it |
| `'header++'` | Header + first paragraph, only if it immediately follows the header |
| `int` (e.g. `30`) | First *n* lines, regardless of structure |
All levels return `''` if the content is empty or (for header levels) has no leading header.


In [ ]:
#| echo
cnt = '#### test\n> Paragraph 1.\nLine.\n\nParagraph 2\n```python\na=13\n```'
test_eq(clamp_note(cnt, 'header'), '#### test')

test_eq(clamp_note(cnt, 'header+'), '#### test\n> Paragraph 1.')
test_eq(clamp_note(cnt, 'header++'), '#### test\n> Paragraph 1.\nLine.')

### message extended

Extends the base `Msg` (`adict`) type with `MsgX`, a subclass that adds convenience properties for inspecting message metadata.
`msgx()` casts any `Msg` in-place to `MsgX` (zero-copy, via `__class__` reassignment). `MsgX` then exposes `.exported_x` and `.hidden_x` as properties wrapping `is_exported`/`is_hidden`.


In [ ]:
import fastcore.basics
from fastcore.basics import adict
adict??
AD??


````python
class adict(dict):
    "`dict` subclass that also provides access to keys as attrs"
    def __getattr__(self,k): return self[k] if k in self else stop(AttributeError(k))
    def __setattr__(self, k, v): (self.__setitem__,super().__setattr__)[k[0]=='_'](k,v)
    def __dir__(self): return super().__dir__() + list(self.keys())
````

**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`


````python
class AttrDict(adict):
    "`dict` subclass that also provides access to keys as attrs, and has a pretty markdown repr"
    def _repr_markdown_(self):
        import pprint
        return f'```python\n{pprint.pformat(self, indent=2)}\n```'

    def copy(self): return AttrDict(**self)
````

**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`

In [ ]:
msg = await read_msg()
test_is(isinstance(msg, adict), True)
test_is(isinstance(msg, Msg), False)
msg

{'idle_evt': '<asyncio.locks.Event object at 0x7afac5742630 [set]>',
 'id': '_eb1a767c',
 'time_run': '2026-05-18T09:31:20.003081+00:00',
 'is_exported': 0,
 'skipped': 1,
 'bookmark': None,
 'i_collapsed': 0,
 'o_collapsed': 0,
 'heading_collapsed': False,
 'i_clamp': False,
 'o_clamp': False,
 'pinned': 0,
 'run': False,
 'oob': None,
 'scroll': None,
 'order': 21,
 'content': 'import fastcore.basics\nfrom fastcore.basics import adict\nadict??\nAD??',
 'output': '\n````python\nclass adict(dict):\n    "`dict` subclass that also provides access to keys as attrs"\n    def __getattr__(self,k): return self[k] if k in self else stop(AttributeError(k))\n    def __setattr__(self, k, v): (self.__setitem__,super().__setattr__)[k[0]==\'_\'](k,v)\n    def __dir__(self): return super().__dir__() + list(self.keys())\n````\n\n**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`\n\n````python\nclass AttrDict(adict):\n    "`dict` subclass that also provides access to keys as attrs, 

In [ ]:
#| export
class MsgX(Msg): ...

In [ ]:
msg.__class__ = MsgX
test_is(isinstance(msg, MsgX), True)

In [ ]:
#| export
def msgx(msg:adict) -> None:
    "Return `msg` cast to MsgX."
    msg.__class__ = MsgX
    return msg

In [ ]:
#| export
@patch(as_prop=True)
def exported_x(self:MsgX) -> bool: return is_exported(self)
@patch(as_prop=True)
def hidden_x(self:MsgX) -> bool: return is_hidden(self)
@patch(as_prop=True)
def echo_x(self:MsgX) -> bool: return is_echo(self)
@patch(as_prop=True)
def header_x(self:MsgX) -> bool: return is_header(self)

What does `msgx` do? Critique.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`msgx` casts an `adict` message to `MsgX` in-place by reassigning `__class__`, then returns it. This is a fast, zero-copy type "upgrade" that lets you use `MsgX` properties/methods on an existing dict without creating a new object.

**Critique:**

- The in-place mutation is a side effect that the name and docstring don't make obvious — "cast in-place" hints at it, but callers might be surprised their original `msg` is also now a `MsgX`
- The type annotation `msg:adict` is slightly misleading since it'll work on any dict subclass (e.g. plain `adict`, `AD`, etc.)
- No guard against passing a `msg` that's already a `MsgX` — harmless, but worth knowing
- It's essentially a one-liner that could be inlined wherever needed; whether it deserves its own function depends on how often it's called

<details class='token-usage-details'><summary>$0.0215</summary>

`total=13,588 | in=13,365 | out=223 | cached=80.7% | $0.0215`

</details>


### extended `find_msgs`
`find_msgs` returning MsgX.

In [ ]:
#| export
@delegates(find_msgs)
async def find_msgs_x(*args, **kwargs) -> list[MsgX]:
    "find_msgs returning MsgX instances."
    ms = await find_msgs(*args, **kwargs)
    for m in ms: m.__class__ = MsgX
    return ms

## dialog utils
Utilities for treating dialogs as first-class objects: resolving their backing files and Python modules, and introspecting runtime state such as cell outputs.

### dialog path
Resolves the filesystem path of a solveit dialog's backing `.ipynb` notebook file.

In [ ]:
#| export
def nb_path(dname:str=''): return (Path.home()/(dname or find_dname()).removeprefix('/')).with_suffix('.ipynb')

In [ ]:
_ = nb_path()
test_is(_.exists(), True)
_

Path('/app/data/prj/pote/nbs/01_dialog.ipynb')

In [ ]:
test_eq(nb_path('/a/b/c'), Path.home()/'a/b/c.ipynb')

### dialog module
Resolves the nbdev module name declared in a dialog or solo module if not found, enabling other tools to know which Python module a dialog exports to (if using `dlg_export`).

In [ ]:
#| export
_exp_pat =r'^\s*#\s*\|\s*default_exp\s+(\w+)'
async def dlg_module(dname:str='', solo_nb:bool=False):
    "Finds the nbdev default_exp in a dialog, or the solo module name if not nbdev"
    from nbdev.config import is_nbdev
    if not is_nbdev(p := nb_path(dname)) or solo_nb: return p.stem
    msgs = await find_msgs(_exp_pat, dname=(dname or find_dname()), limit=1, include_output=False, include_meta=False)
    if msgs: return msgs[0].content.strip().split()[-1]

In [ ]:
test_eq(await dlg_module(), 'dialog')
test_eq(await dlg_module('/a/b/c'), 'c')

### dialog export
Triggers nbdev export for a dialog, compiling its exported cells into the corresponding Python module on disk.

In [ ]:
#| export
async def dlg_export(dname:str='', solo_nb:bool=False):
    "Export dialog `dname` to Python module"
    from nbdev.export import nb_export
    from nbdev.config import get_config, is_nbdev
    p = nb_path(dname)
    if not (_is := is_nbdev(p)): solo_nb = True
    mod = await dlg_module(dname, solo_nb=solo_nb)
    lib_path = p.parent if solo_nb else get_config(p).lib_path if _is else Path('.')
    if solo_nb: nb_export(p, lib_path=lib_path, name=mod, solo_nb=solo_nb)
    else: nb_export(p)
    return (lib_path/f"{mod}.py").exists()

In [ ]:
mod_p = (get_config().lib_path/(await dlg_module())).with_suffix('.py')
if mod_p.exists(): mod_p.unlink()

test_is(await dlg_export(), True)
test_is(mod_p.exists(), True)

### dialog link
Generates a clickable HTML hyperlink to any solveit dialog by path, for use in notes and outputs.

In [ ]:
#| export
def get_dialog_link(
    path:str # Path to dialog (e.g. '/aai-ws/dialoghelper/nbs/00_core')
) -> str:
    "Return an HTML link to open a dialog in Solveit."
    from dialoghelper.core import dialog_link
    return dialog_link(path).data

In [ ]:
get_dialog_link('/prj/dutil/nbs/00_dutil')

'<a href="/dialog_?name=prj%2Fdutil%2Fnbs%2F00_dutil" target="_blank">prj/dutil/nbs/00_dutil</a>'

### empty .ipynb
Empty .ipynb file.

In [ ]:
#| export
def empty_dialog_nb() -> str:
    "Return a minimal dialog notebook json"
    return '''
{
"cells": [],
"metadata": {
"solveit_dialog_mode": "learning",
"solveit_ver": 2
},
"nbformat": 4,
"nbformat_minor": 5
}
'''

### message id
Exposes the id of the currently executing message, enabling code to self-reference its own position in the dialog.

In [ ]:
#| export
async def get_msg_id():
    "Get current message id."
    return (await read_msg(0)).id

In [ ]:
test_eq(await get_msg_id(), '_4d300912')

### get output
Retrieves the rendered output string of any message by id, enabling programmatic inspection of what a cell produced.

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

In [ ]:
2;

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

In [ ]:
2

2

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '2')

In [ ]:
display(HTML('asdf'))

HTML(asdf)

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, 'asdf')

In [ ]:
display(HTML('<ul><li>one</li></ul>'))

HTML(<ul><li>one</li></ul>)

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '<ul><li>one</li></ul>')

In [ ]:
display(Markdown('### hdr3\n\n**bold**'))

<div class="prose" markdown="1">

### hdr3

**bold**

</div>

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '<div class="prose" markdown="1">\n\n### hdr3\n\n**bold**\n\n</div>')

In [ ]:
msg.output

'<div class="prose" markdown="1">\n\n### hdr3\n\n**bold**\n\n</div>'

a note

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

a prompt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

a response

In [ ]:
promptid = (msg := await read_msg()).id
test_eq(msg.output, 'a response')

In [ ]:
msgid = (msg := await read_msg()).id
test_eq(msg.output, '')

In [ ]:
#| export
@delegates(read_msg)
async def get_output(id:str=None, **kwargs) -> list[str]:
    msg = await read_msg(0, id=id, **kwargs)
    return msg.output

In [ ]:
test_eq(await get_output(promptid), 'a response')

In [ ]:
FC.noop?

````python
def noop(
    x:NoneType=None, args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):

````



````
Do nothing
````



**File:** `/usr/local/lib/python3.12/site-packages/fastcore/imports.py`

**Type:** function

In [ ]:
(await read_msg()).output

'````python\ndef noop(\n    x:NoneType=None, args:VAR_POSITIONAL, kwargs:VAR_KEYWORD\n):\n\n````\n\n\n\n````\nDo nothing\n````\n\n\n\n**File:** `/usr/local/lib/python3.12/site-packages/fastcore/imports.py`\n\n**Type:** function'

### output formatting
Converts Python objects into a JSON-encoded display payload — capturing all available MIME representations (plain text, HTML, markdown) — for programmatic use outside the normal cell output pipeline.

In [ ]:
#| exporti
_format = get_ipython().display_formatter.format if in_dialog() else None

In [ ]:
_format(...)

({'text/plain': 'Ellipsis'}, {})

In [ ]:
_format(HTML('a'))

({'text/plain': 'HTML(a)', 'text/html': 'a'}, {})

In [ ]:
_format(Div('a'))

({'text/plain': '<div>a</div>',
  'text/html': '<div>a</div>',
  'text/markdown': '<div class="prose" markdown="1">\n\n```html\n<div>a</div>\n\n```\n\n</div>'},
 {})

In [ ]:
#| export
delegates(_format)
def format_output(o, **kwargs):
  d, md = _format(o, **kwargs) if _format else ('', {})
  return json.dumps([{"data": d, "metadata": md, "output_type": "display_data"}])

In [ ]:
format_output('')

'[{"data": {"text/plain": "\'\'"}, "metadata": {}, "output_type": "display_data"}]'

In [ ]:
format_output('a')

'[{"data": {"text/plain": "\'a\'"}, "metadata": {}, "output_type": "display_data"}]'

In [ ]:
format_output(Pre('a\nb\nc'))

'[{"data": {"text/plain": "<pre>a\\nb\\nc</pre>", "text/html": "<pre>a\\nb\\nc</pre>", "text/markdown": "<div class=\\"prose\\" markdown=\\"1\\">\\n\\n```html\\n<pre>a\\nb\\nc</pre>\\n```\\n\\n</div>"}, "metadata": {}, "output_type": "display_data"}]'

In [ ]:
format_output(Div('a'), exclude=['text/html'])

'[{"data": {"text/plain": "<div>a</div>", "text/markdown": "<div class=\\"prose\\" markdown=\\"1\\">\\n\\n```html\\n<div>a</div>\\n\\n```\\n\\n</div>"}, "metadata": {}, "output_type": "display_data"}]'

### markdown splitter
Splits a markdown string into its constituent blocks, separating fenced code blocks from surrounding prose, so callers can process or transform each block type independently.

In [ ]:
def split_md_blocks(md):
    "Split markdown string into list of fenced and non-fenced block strings"
    return [b for b in re.split(r'(^```\w*\n.*?^```)', md, flags=re.DOTALL|re.MULTILINE) if b.strip()]

In [ ]:
for _ in split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text"): print(_, '\n---')

# Hello

Some text

 
---
```python
print('hi')
``` 
---


More text 
---


The idiomatic mistletoe way is to walk the parsed AST and use a renderer for non-code blocks:

In [ ]:
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with HtmlRenderer() as renderer:
        doc = Document(md)
        return [f'```{c.language}\n{c.children[0].content}```' if isinstance(c, CodeFence)
                else renderer.render(c) for c in doc.children]

In [ ]:
split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text")

['<h1>Hello</h1>',
 '<p>Some text</p>',
 "```python\nprint('hi')\n```",
 '<p>More text</p>']

The key ideas here are: `Document` parses into an AST whose `.children` are block-level tokens; `CodeFence` tokens carry `.language` and their code content nested as `children[0].content`; everything else gets rendered to HTML via the renderer context. This gives you a starting point to explore other token types like `Heading`, `Paragraph`, `List`, etc.

We can swap to `MarkdownRenderer` which round-trips tokens back to raw markdown.

In [ ]:
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with MarkdownRenderer() as renderer:
        doc = Document(md)
        return [renderer.render(c) for c in doc.children]

In [ ]:
split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text")

['# Hello\n',
 '\n',
 'Some text\n',
 '\n',
 "```python\nprint('hi')\n```\n",
 '\n',
 'More text\n']

Collapse consecutive non-fenced code blocks.

In [ ]:
#| export
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with MarkdownRenderer() as renderer:
        doc = Document(md)
        blocks,buf = [],[]
        for c in doc.children:
            if isinstance(c, CodeFence):
                if buf: blocks.append(''.join(buf)); buf = []
                blocks.append(renderer.render(c))
            else: buf.append(renderer.render(c))
        if buf: blocks.append(''.join(buf))
        return blocks

In [ ]:
test_eq(split_md_blocks("# Hello\n\nSome text\n\n```python\nprint('hi')\n```\n\nMore text"),
    ['# Hello\n\nSome text\n\n', "```python\nprint('hi')\n```\n", '\nMore text\n'])

test_eq(split_md_blocks("No code blocks at all."), 
        ['No code blocks at all.\n'])

test_eq(split_md_blocks("```python\nonly_code()\n```"), 
        ['```python\nonly_code()\n```\n'])

test_eq(split_md_blocks("Before\n```python\ncode1()\n```\n```bash\ncode2\n```\nAfter"), 
        ['Before\n', '```python\ncode1()\n```\n', '```bash\ncode2\n```\n', 'After\n'])

test_eq(split_md_blocks("Text\n```\nno language\n```\nMore"), 
        ['Text\n', '```\nno language\n```\n', 'More\n'])

test_eq(split_md_blocks("A\n```python\ncode\n```\nB\n```js\nmore\n```\nC"), 
        ['A\n', '```python\ncode\n```\n', 'B\n', '```js\nmore\n```\n', 'C\n'])

### token count

In [ ]:
def count_tokens(src, model='cl100k_base'):
    "Count tokens in `src` using `model` encoding"
    import tiktoken
    return len(tiktoken.get_encoding(model).encode(src))

In [ ]:
count_tokens("def foo(x:int) -> bool: return x > 0")

13

## Code message disclosure
Progressive disclosure of Python source code at three levels: signatures only (level 1), signatures with docstrings and comments (level 2), and full source (level 3). Per-cell, preserving message boundaries and ids.

### disclose code

See <a href="/dialog_?name=prj%2Fpote%2Fexplorer%2Fcode_disclosure%2Fcode_disclosure_v1" target="_blank">prj/pote/explorer/code_disclosure/code_disclosure_v2</a>

In [ ]:
#| export
def _is_docstring(n): return isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant) and isinstance(n.value.value, str)

def _reindent(src, n): return '\n'.join(' '*n + l for l in src.splitlines())

def _postprocess(s):
    "Collapse triple+ blank lines and hoist lone `...` body onto the signature line"
    s = re.sub(r'\n{3,}', '\n\n', s)
    return re.sub(r':\n[ \t]+\.\.\.', ': ...', s)

_compound_types = (ast.If, ast.For, ast.While, ast.With, ast.Try, ast.Match)

In [ ]:
#| export
def _strip_private(tree):
    "Remove private (underscore-prefixed) top-level nodes and class methods from `tree` in-place"
    is_priv = lambda n: getattr(n, 'name', '').startswith('_')
    tree.body = [n for n in tree.body if not is_priv(n)]
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            node.body = [n for n in node.body if not is_priv(n)]
    return tree

In [ ]:
#| export
def stub_fn(atok, fn, level=1):
    "Return stubbed source for function `fn` at disclosure `level`"
    if level == 3: return atok.get_text(fn)
    start_line = fn.decorator_list[0].lineno if fn.decorator_list else fn.lineno
    body_start = fn.body[0].lineno - start_line
    if level == 1 or body_start == 0:
        n = copy.deepcopy(fn)
        n.body = [ast.Expr(ast.Constant(...))]
        return _reindent(ast.unparse(n), fn.col_offset)
    indent = ' ' * fn.body[0].col_offset
    lines = atok.get_text(fn).splitlines()
    if _is_docstring(fn.body[0]):
        doc_end = fn.body[0].end_lineno - start_line + 1
        return '\n'.join(lines[:body_start] + lines[body_start:doc_end] + [f'{indent}...'])
    return '\n'.join(lines[:body_start] + [f'{indent}...'])

In [ ]:
_src = 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20'
_atok = asttokens.ASTTokens(_src, parse=True)
_fn = _atok.tree.body[0]
test_eq(stub_fn(_atok, _fn, 1), 'def f(a: int) -> int:\n    ...')
test_eq(stub_fn(_atok, _fn, 2), 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    ...')
test_eq(stub_fn(_atok, _fn, 3), _src)

In [ ]:
#| export
def _class_chunk(atok, node, level):
    "Return disclosed source for class `node` at `level`"
    if level == 3: return atok.get_text(node)
    start_line = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_start = node.body[0].lineno - start_line
    lines = atok.get_text(node).splitlines()
    hdr = '\n'.join(lines[:body_start])
    body = []
    for i,n in enumerate(node.body):
        if isinstance(n, functypes): body.append(stub_fn(atok, n, level))
        elif isinstance(n, ast.Expr):
            if i == 0 and _is_docstring(n) and level >= 2: body.append(' '*n.col_offset + atok.get_text(n))
        else: body.append(' '*n.col_offset + atok.get_text(n))
    return hdr + '\n' + '\n'.join(body)

def _compound_chunk(atok, node, level):
    "Return disclosed source for compound statement `node` at `level`"
    if level == 3: return atok.get_text(node)
    text = atok.get_text(node)
    fns = sorted([n for n in ast.walk(node) if n is not node and isinstance(n, (*functypes, ast.ClassDef))],
                 key=lambda n: (n.lineno, n.col_offset), reverse=True)
    for n in fns:
        stub = stub_fn(atok, n, level) if isinstance(n, functypes) else _class_chunk(atok, n, level)
        lines = stub.splitlines()
        if lines[0].startswith(' ' * n.col_offset): lines[0] = lines[0][n.col_offset:]
        text = text.replace(atok.get_text(n), '\n'.join(lines), 1)
    return text

In [ ]:
#| export
def disclose(src:str, level=1, strip_private=False) -> str:
    "Disclose Python source at `level` (1=sigs, 2=sigs+docs, 3=full)"
    if not src.strip(): return ''
    try:
        atok = asttokens.ASTTokens(src, parse=True)
        if strip_private: _strip_private(atok.tree)
        chunks = []
        for i,node in enumerate(atok.tree.body):
            if isinstance(node, functypes): chunks.append(stub_fn(atok, node, level))
            elif isinstance(node, ast.ClassDef): chunks.append(_class_chunk(atok, node, level))
            elif isinstance(node, _compound_types): chunks.append(_compound_chunk(atok, node, level))
            elif isinstance(node, ast.Expr) and not (i == 0 and _is_docstring(node) and level >= 2): pass
            elif isinstance(node, (ast.Import, ast.ImportFrom)) and level == 1: pass
            else: chunks.append(atok.get_text(node))
        return _postprocess('\n\n'.join(chunks))
    except SyntaxError as e:
        warnings.warn(f'Syntax error in source, returning raw: {e}')
        return src

In [ ]:
test_eq(disclose('', 1), '')
test_eq(disclose("def bar(): pass", 1), "def bar(): ...")
test_eq(disclose("def bar(): pass", 2), "def bar(): ...")
test_eq(disclose("def bar(): pass", 3), "def bar(): pass")

_s = 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20'
test_eq(disclose(_s, 1), "def f(a: int) -> int: ...")
test_eq(disclose(_s, 2), 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    ...')
test_eq(disclose(_s, 3), _s)

### dialog and notebook disclosure
Integrates per-cell `disclose` with solveit dialog and notebook file workflows. Three inclusion tiers: `'exported'` (module API), `'echo'` (API + doc examples), `'all'`.

In [ ]:
#| export
_include_filter = dict(
    exported=lambda m: is_exported(m),
    echo=lambda m: is_exported(m) or is_echo(m),
    all=lambda m: True)

def _compact(s): return re.sub(r'\n{2,}', '\n', s).strip()

def _xml_cell(content, **attrs):
    attr_str = ''.join(f' {k}="{v}"' for k,v in attrs.items())
    return f'<cell{attr_str}>{content}</cell>'

In [ ]:
#| export
async def dialog_code(dname='', include='all'):
    "Return list of (id, source) tuples for code cells filtered by `include`"
    msgs = await find_msgs_x(dname=dname, msg_type='code', include_output=False)
    return [(m.id, m.content) for m in msgs if _include_filter[include](m)]

In [ ]:
#| export
def _disclose_srcs(srcs, level=1, strip_private=False, fmt='text', dname=''):
    "Disclose list of (id, source) tuples at `level`"
    chunks = [(mid, disclose(src, level, strip_private)) for mid,src in srcs]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(escape(src), id=mid) for mid,src in chunks)
        return f'<dialog level="{level}" path="{dname or "."}">\n{cells}\n</dialog>'
    if fmt == 'json':
        cells = [dict(id=mid, source=src) for mid,src in chunks]
        return json.dumps(dict(level=level, path=dname or '.', cells=cells), indent=2)
    return _compact('\n\n'.join(src for _,src in chunks))

async def disclose_dialog(dname='', level=1, include='all', strip_private=False, fmt='text'):
    "Disclose dialog `dname` at `level` filtering by `include`"
    srcs = await dialog_code(dname=dname, include=include)
    return _disclose_srcs(srcs, level=level, strip_private=strip_private, fmt=fmt, dname=dname)

def disclose_nb(path, level=1, strip_private=False, fmt='text'):
    "Return disclosed source of all code cells in notebook at `path`"
    nb = json.loads(Path(path).read_text())
    srcs = [(c.get('id',''), c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for c in nb['cells'] if c['cell_type'] == 'code']
    return _disclose_srcs(srcs, level=level, strip_private=strip_private, fmt=fmt, dname=str(path))

### tests

In [ ]:
def _test_dlg(cs, fp='test_disclose.ipynb'):
    nb = writes_json(new_notebook(cells=[new_code_cell(c) for c in cs]))
    (p := Path(fp)).write_text(nb)
    return str(p)

test_eq(disclose('', 1), '')
test_eq(disclose('', 2), '')
test_eq(disclose('', 3), '')

test_eq(disclose('(1+2)', 1), '')
test_eq(disclose('(1+2)', 2), '')
test_eq(disclose('(1+2)', 3), '')

src = 'if a: import A\nelse: import B'
test_eq(disclose(src, 1), src)
test_eq(disclose(src, 2), src)
test_eq(disclose(src, 3), src)

_s_mods = "x = 5\ny:int = 10\ntype Alias = list[int]"
test_eq(disclose(_s_mods, 1), 'x = 5\n\ny:int = 10\n\ntype Alias = list[int]')
test_eq(disclose(_s_mods, 2), disclose(_s_mods, 1))
test_eq(disclose(_s_mods, 3), disclose(_s_mods, 1))

In [ ]:
_s_moddoc = '"""This module does X."""\n\ndef foo(): pass'
test_eq(disclose(_s_moddoc, 1), "def foo(): ...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\n\ndef foo(): ...')
test_eq(disclose(_s_moddoc, 3), '"""This module does X."""\n\ndef foo(): pass')

_s_fn = "def foo(a:int, b:str='hi') -> bool: return True"
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool: ...")
test_eq(disclose(_s_fn, 3), _s_fn)

_s_async = "async def fetch(url:str) -> bytes:\n    'Fetch url.'\n    pass"
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes: ...")

_s_deco1 = "@patch\ndef foo(self:MyClass, x:int): pass"
test_eq(disclose(_s_deco1, 1), "@patch\ndef foo(self: MyClass, x: int): ...")

_s_deco2 = "@log\n@patch\ndef foo(self:MyClass, x:int): pass"
test_eq(disclose(_s_deco2, 1), "@log\n@patch\ndef foo(self: MyClass, x: int): ...")

_s_doc = '\ndef add(\n    a:int, # first\n    b:int=0, # second\n) -> int:\n    "Sum two numbers."\n    return a+b\n'
test_eq(disclose(_s_doc, 1), "def add(a: int, b: int=0) -> int: ...")
test_eq(disclose(_s_doc, 2), 'def add(\n    a:int, # first\n    b:int=0, # second\n) -> int:\n    "Sum two numbers."\n    ...')
test_eq(disclose(_s_doc, 3), _s_doc.strip())
test_is('return a+b' in disclose(_s_doc, 2), False)
test_is('return a+b' in disclose(_s_doc, 3), True)

In [ ]:
_s_cls = '\nclass Foo(Base):\n    "A foo."\n    def __init__(self,\n                 x:int, # x val\n                 y:str=\'hi\'): # y val\n        "Init."\n        self.x,self.y = x,y\n    def bar(self) -> bool:\n        "Check bar."\n        return self.x > 0\n    def _priv(self): pass\n'
_cls_l1 = 'class Foo(Base):\n    def __init__(self, x: int, y: str=\'hi\'): ...\n    def bar(self) -> bool: ...\n    def _priv(self): ...'
_cls_l2 = 'class Foo(Base):\n    "A foo."\n    def __init__(self,\n                 x:int, # x val\n                 y:str=\'hi\'): # y val\n        "Init."\n        ...\n    def bar(self) -> bool:\n        "Check bar."\n        ...\n    def _priv(self): ...'
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())

_s_cls_assigns = '\nclass Config:\n    "App config."\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self): self.debug = False\n'
test_eq(disclose(_s_cls_assigns, 1), 'class Config:\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self): ...')
test_eq(disclose(_s_cls_assigns, 2), 'class Config:\n    "App config."\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self): ...')
test_eq(disclose(_s_cls_assigns, 3), _s_cls_assigns.strip())

_s_dc = '\nfrom dataclasses import dataclass\n\n@dataclass\nclass Point:\n    "A 2D point."\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        "Distance from origin."\n        return (self.x**2 + self.y**2)**0.5\n'
test_eq(disclose(_s_dc, 1), "@dataclass\nclass Point:\n    x: float\n    y: float = 0.0\n    def dist(self) -> float: ...")
test_eq(disclose(_s_dc, 2), "from dataclasses import dataclass\n\n@dataclass\nclass Point:\n    \"A 2D point.\"\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        \"Distance from origin.\"\n        ...")
test_eq(disclose(_s_dc, 3), _s_dc.strip())

_s_cond = '\nimport sys\n\nif sys.version_info >= (3, 12):\n    def new_feature(x:int) -> str: return str(x)\nelse:\n    def new_feature(x:int) -> str: return repr(x)\n\ndef always_here(): pass\n'
test_eq(disclose(_s_cond, 1), 'if sys.version_info >= (3, 12):\n    def new_feature(x: int) -> str: ...\nelse:\n    def new_feature(x: int) -> str: ...\n\ndef always_here(): ...')
test_eq(disclose(_s_cond, 2), 'import sys\n\nif sys.version_info >= (3, 12):\n    def new_feature(x: int) -> str: ...\nelse:\n    def new_feature(x: int) -> str: ...\n\ndef always_here(): ...')
test_eq(disclose(_s_cond, 3), _s_cond.strip())

In [ ]:
_s_imp = '\n"""A module."""\nimport os, sys\nfrom pathlib import Path\n\ndef foo(): pass\n'
test_eq(disclose(_s_imp, 1), "def foo(): ...")
test_eq(disclose(_s_imp, 2), '"""A module."""\n\nimport os, sys\n\nfrom pathlib import Path\n\ndef foo(): ...')
test_eq(disclose(_s_imp, 3), '"""A module."""\n\nimport os, sys\n\nfrom pathlib import Path\n\ndef foo(): pass')

_s_tricky = '''
def foo():
    """
    Example::

        :\\n    ...

    ```python
    x = 1
    ```
    """
    pass

def bar(): pass

def baz():
    """Has "quotes" and \\'escaped\\' stuff."""
    pass
'''
test_eq('```python' in disclose(_s_tricky, 2), True)
test_eq('pass' in disclose(_s_tricky, 3), True)

_test_sp = '\ndef pub(x:int) -> int:\n    "public fn"\n    return _helper(x)\n\ndef _helper(x): return x+1\n\nclass Foo:\n    def pub_method(self): return 1\n    def _priv_method(self): return 2\n'
test_eq(disclose(_test_sp, 1, strip_private=True), 'def pub(x: int) -> int: ...\n\nclass Foo:\n    def pub_method(self): ...')
test_eq(disclose(_test_sp, 2, strip_private=True), 'def pub(x:int) -> int:\n    "public fn"\n    ...\n\nclass Foo:\n    def pub_method(self): ...')

cs = ['a=10', 'def f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20']
test_dname = _test_dlg(cs)
test_eq(disclose_nb(test_dname, level=1), 'a=10\ndef f(a: int) -> int: ...')
test_eq(disclose_nb(test_dname, level=2), 'a=10\ndef f(a:int # arg\n) -> int: # ret\n    "a doc"\n    ...')
test_eq(disclose_nb(test_dname, level=3), 'a=10\ndef f(a:int # arg\n) -> int: # ret\n    "a doc"\n    return a+20')

### Code disclosure: rationale, decisions, current state

Progressive disclosure of Python source code at three levels modelled on `.pyi` stub files. Per-cell, preserving message boundaries and dialog narrative structure.

| Level | Description | Analogy |
|---|---|---|
| 1 | Sigs only, no docs, no imports, no comments | Minified `.pyi` |
| 2 | Sigs + docstrings + inline param comments + imports | Standard `.pyi` |
| 3 | Full source, untouched | `.py` |

**Implementation**: `asttokens` maps AST nodes to original source spans. Level 1 uses `ast.unparse` (comments stripped naturally). Level 2 lifts original source lines (comments preserved). `_postprocess` hoists lone `...` bodies and collapses triple+ blank lines.

**Design decisions**:
- Imports stripped at level 1 only (executability at levels 2/3)
- Bare `ast.Expr` stripped at all levels except module docstring at level ≥ 2
- `strip_private` flag removes `_`-prefixed functions/methods (safe at levels 1/2)
- `_compact` collapses inter-cell blank lines for text output format
- Three inclusion tiers via `_include_filter`: `'exported'`, `'echo'`, `'all'`

**Known limitations**:
- `_postprocess` ellipsis hoisting can match inside docstrings containing `:\n    ...` patterns (rare)
- Side-effect expressions (`print(...)`, `setup(...)`) are stripped — indistinguishable from pure expressions by AST alone
- `_compound_chunk` uses `text.replace()` which could misfire if two functions inside a compound have identical source text
- `@patch` consolidation and `@delegates` expansion not yet implemented

**Token ratios** (this dialog, exported cells): level 1 ~22%, level 2 ~43%, level 3 ~105% of raw.

## distill
Utilities for reading and writing message content across sessions.

### filter messages v0
Filters a list of messages down to those relevant for export or documentation: exported (non-hidden) code cells, note headers, and cells with an echo directive.

In [ ]:
#| echo
async def filter_msgs_v0(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Return notes, exported messages not hidden and code cells that have echo directive."
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return [m for m in msgs if (is_exported(m) and not is_hidden(m)) or is_header(m) or is_echo(m)]

In [ ]:
test_export_dname = find_dname('data/test_export')
test_export_dname

'/prj/pote/nbs/data/test_export'

In [ ]:
test_msgs = await find_msgs_x(dname=test_export_dname, include_skipped=True)

In [ ]:
fed_test_msgs = await filter_msgs_v0(test_msgs)
test_eq(len(fed_test_msgs), 9)

In [ ]:
msgs = await find_msgs_x(include_skipped=True)
first_idx = await msg_idx('_cf210ba7')  # # dialog
last_idx = await msg_idx('_842dc9a8')  # async def importdlg(

In [ ]:
dlog_msgs = msgs[first_idx:last_idx]  # messages in main section
len(dlog_msgs), len(await filter_msgs_v0(dlog_msgs))

(235, 94)

### digest dialog
Format a filtered digest of dialog messages for AI consumption. Supports `xml` (preferred), `md`, and `json` formats. Headers are progressively disclosed via `clamp_note` to keep digests compact. YAML frontmatter is prepended via `mk_frontmatter`.


In [ ]:
#| export
_pmark = '**🤖Reply🤖**<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n'

In [ ]:
#| export
def _mk_tag(m, mtyp, ids):
    # parts = ([f'#{m.id}'] if ids else []) + [mtyp] + (['echo'] if echo else []) + (['skipped'] if m.skipped else []) + (['pinned'] if m.pinned else [])
    parts = ([f'#{m.id}'] if ids else []) + [mtyp]
    return ' '.join(parts)

In [ ]:
#| export
def output2md(c, langid='output'):
    return f'''```{langid}\n{c}\n```''' if c else ''

In [ ]:
test_eq(_mk_tag(MsgX(id='a'), 'code', ['a']), '#a code')

In [ ]:
#| export
def strip_directives(c): return ('\n'.join(l for l in c.splitlines() if not l.startswith('#|'))).strip()

In [ ]:
#| export
_striptag_pats = (
    re.compile(r'^#\|\W*\w+: \w+\W*$\n?', re.MULTILINE),
    re.compile(r'^<!-- \w+: \w+ -->\s*\n?', re.MULTILINE))

def strip_tags(c):
    for pat in _striptag_pats: c = pat.sub('', c)
    return c.strip()

In [ ]:
#| export
def _xml_msg(m:MsgX, ids, output, hdr_clamp):
    mtyp = m.msg_type
    c, attrs = '', (f' id="{m.id}"' if ids else '') + f' type="{mtyp}"'
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = escape(strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content))
            if c: c = f'<content>{c}</content>'
        case 'code':
            src = escape(strip_directives(m.content))
            out = f'\n  <output>{escape(m.output)}</output>' if output and m.output else ''
            if src or out: c = f'\n  <source>{src}</source>{out}\n'
        case 'prompt':
            p = escape(strip_tags(strip_directives(m.content)))
            r = f'\n  <response>{escape(m.output)}</response>' if m.output else ''
            if p or r: c = f'<prompt>{p}</prompt>{r}'
        case 'raw':
            c = escape(strip_tags(strip_directives(m.content)))
            if c: c = f'<msg{attrs}><content>{c}</content></msg>'
    return f'<msg{attrs}>{c}</msg>' if c else ''

In [ ]:
#| export
def _md_msg(m, ids, output, hdr_clamp):
    mtyp, c = m.msg_type, None
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            c = f'```python\n{strip_directives(m.content)}\n```'
            if output and m.output: c += f'\n{output2md(m.output)}'
        case 'prompt':
            c = strip_tags(strip_directives(m.content))
            if output and m.output: c += f'\n{output2md(m.output, 'response')}'
        case 'raw': c = strip_tags(strip_directives(m.content))
    return f'<!-- {_mk_tag(m, mtyp, ids)} -->\n{c}\n' if c else ''

In [ ]:
#| export
def _json_msg(m, expd, echo, ids, output, hdr_clamp):
    mtyp = m.msg_type
    d = dict(type=mtyp)
    if ids: d['id'] = m.id
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            d['content'] = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            d['source'] = strip_directives(m.content)
            if output and m.output: d['output'] = m.output
        case 'prompt':
            d['prompt'] = strip_tags(strip_directives(m.content))
            d['response'] = m.output
        case 'raw': d['content'] = strip_tags(strip_directives(m.content))
    return d

In [ ]:
#| export
_convs = dict(
    xml="msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs",
    json="msgs is a list of objects with type and id fields; code objects have source and output fields; prompt objects have prompt and response fields; others have content",
    md="<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response")

In [ ]:
#| export
def mk_frontmatter(path='', description='', n_msgs=0, n_exported=0, fmt='xml'):
    from datetime import datetime, timezone
    return ("---\n"
"digest: solveit-dialog\n"
f"path: {path}\n"
f"exported_at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
f"description: {description!r}\n"
f"stats: {{total: {n_msgs}, exported: {n_exported}}}\n"
f"conventions: '{_convs[fmt]}'\n"
"---\n"
)

In [ ]:
#| export
def exportmsgs(
    msgs:list[Msg],
    hdr_clamp='header+',  # header level
    output=True,  # include code messages output
    ids=True,  # include message ids in cell comments
    frontmatter:dict=None,  # if present, call `mk_frontmatter` with this dict as args
    fmt='xml',  # 'xml'|'md'|'json'
):
    "Export messages as a digest for AI consumption"
    fn = {'xml': _xml_msg, 'md': _md_msg, 'json': _json_msg}[fmt]
    parts = [p for m in msgs if (p := fn(m, ids, output, hdr_clamp))]
    if fmt == 'json': res = json.dumps({'msgs': parts}, indent=2)
    else:
        body = '\n'.join(parts)
        res = f'<digest>\n{body}\n</digest>' if fmt=='xml' else body
    if frontmatter:
        frontmatter.update(n_exported=len(parts))
        fm = mk_frontmatter(**frontmatter)
    return f"{fm}\n{res}" if frontmatter else res

In [ ]:
fmt = 'xml'
s = exportmsgs(test_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=test_export_dname, description='A digest for testing purposes', n_msgs=len(test_msgs), fmt=fmt))

print(len(s))
print(s)

1897
---
digest: solveit-dialog
path: /prj/pote/nbs/data/test_export
exported_at: 2026-05-18T09:31:28+00:00
description: 'A digest for testing purposes'
stats: {total: 15, exported: 15}
conventions: 'msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs'
---

<digest>
<msg id="_b6147aa6" type="note"><content>##### Header (exported)</content></msg>
<msg id="_fad144f0" type="note"><content>##### Header w/ paragraph (exported)
Paragraph.
Line 2.</content></msg>
<msg id="_2752a761" type="note"><content>##### Note with code fence (exported, only header)</content></msg>
<msg id="_cdfea4f6" type="note"><content>Note w/out header (not exported)</content></msg>
<msg id="_4e4b6f65" type="code">
  <source># exported code
a = 13; print(a+1)</source>
  <output>14</output>
</msg>
<msg id="_fde18916" type="code">
  <source># exported code, echo
test_eq(a, 13)</source>
</msg>
<msg id="_44d67d56" type="code">
  <source

In [ ]:
fmt = 'xml'
s = exportmsgs(fed_test_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=test_export_dname, description='A digest for testing purposes', n_msgs=len(test_msgs), fmt=fmt))

print(len(s))
print(s)

1292
---
digest: solveit-dialog
path: /prj/pote/nbs/data/test_export
exported_at: 2026-05-18T09:31:29+00:00
description: 'A digest for testing purposes'
stats: {total: 15, exported: 9}
conventions: 'msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs'
---

<digest>
<msg id="_b6147aa6" type="note"><content>##### Header (exported)</content></msg>
<msg id="_fad144f0" type="note"><content>##### Header w/ paragraph (exported)
Paragraph.
Line 2.</content></msg>
<msg id="_2752a761" type="note"><content>##### Note with code fence (exported, only header)</content></msg>
<msg id="_4e4b6f65" type="code">
  <source># exported code
a = 13; print(a+1)</source>
  <output>14</output>
</msg>
<msg id="_fde18916" type="code">
  <source># exported code, echo
test_eq(a, 13)</source>
</msg>
<msg id="_868ac4bd" type="code">
  <source># exported, directive
c = 99</source>
</msg>
<msg id="_a607681b" type="prompt"><prompt>a p

In [ ]:
fmt = 'md'
s = exportmsgs(fed_test_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=test_export_dname, description='A digest for testing purposes', n_msgs=len(test_msgs), fmt=fmt))

print(s)

---
digest: solveit-dialog
path: /prj/pote/nbs/data/test_export
exported_at: 2026-05-18T09:31:29+00:00
description: 'A digest for testing purposes'
stats: {total: 15, exported: 9}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_b6147aa6 note -->
##### Header (exported)

<!-- #_fad144f0 note -->
##### Header w/ paragraph (exported)
Paragraph.
Line 2.

<!-- #_2752a761 note -->
##### Note with code fence (exported, only header)

<!-- #_4e4b6f65 code -->
```python
# exported code
a = 13; print(a+1)
```
```output
14
```

<!-- #_fde18916 code -->
```python
# exported code, echo
test_eq(a, 13)
```

<!-- #_868ac4bd code -->
```python
# exported, directive
c = 99
```

<!-- #_a607681b prompt -->
a prompt (exported)
```response
a response
```

<!-- #_e1f61bec raw -->
RAW (exported)

<!-- #_55d40474 raw -->
EXPORTED RAW CONTENT



In [ ]:
fmt = 'xml'
s = exportmsgs(await filter_msgs_v0(dlog_msgs), hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=find_dname(), description='pote.dutil dialog', n_msgs=len(msgs), fmt=fmt))

print(len(s))
print(s[:2000])

44158
---
digest: solveit-dialog
path: /prj/pote/nbs/01_dialog
exported_at: 2026-05-18T09:31:29+00:00
description: 'pote.dutil dialog'
stats: {total: 240, exported: 94}
conventions: 'msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs'
---

<digest>
<msg id="_cf210ba7" type="note"><content># dialog
&gt; dialog provides message and dialog introspection, and import/export utilities for solveit dialogs.</content></msg>
<msg id="_e069f418" type="note"><content>## message utils
Core primitives for working with solveit messages: a `Msg` type alias, utilities for reading nbdev directives and message metadata, progressive disclosure of note content, and an extended `MsgX` subclass with convenience properties.</content></msg>
<msg id="_ea3120ed" type="note"><content>### message tag
Defines `Msg` as a type alias for `adict`, providing a concise, readable type annotation for solveit message objects throughout t

#| export
Format a filtered, clamped digest of dialog messages for AI consumption in other sessions. Three output formats are supported: `xml` (preferred — unambiguous structure), `md`, and `json`, each implemented by a parallel `_xml_msg`/`_md_msg`/`_json_msg` function dispatched by `exportmsgs`.

**Clamping:** note content is progressively disclosed via `clamp_note(level)` — from header-only to header+paragraph — unless the note is fully exported. This keeps digests compact while preserving section structure and intent.

**Frontmatter:** `mk_frontmatter` prepends YAML metadata (path, timestamp, stats, format conventions) so the consuming LLM knows what it's looking at.

**Next.**
Isolate sections and filtered list of messages. Two modes:
- gather list of messages and export as is
- gather list of messages and dependencies (real isolation) so if written in a new dialog will execute correctly. See isolation dialog exploration.

### filter messages v1

In [ ]:
(await read_msg(0, id='_084ce165')).content

"#| exporti\n_exports = {'exporti', 'exports', 'export'}\n_hidden = {'hide'}\n_echo = {'echo', 'echo:'}"

In [ ]:
msgs = await find_msgs(only_exp=True)
test_is('_084ce165' in {_.id for _ in msgs}, False)

#| export
`find_msgs` is a staple of SolveIt, a very powerful introspection tool. But it have some limitations (see above how it doesn't honor explicit `nbdev` export directives) and I need more search granularity (e.g., to improve `exportmsgs`).

We're going to implement `filter_msgs_v1`, a wrapper over `find_msgs_x` with progressively improved search semantics. With no arguments, `filter_msgs_v1` return an empty list.

Let's start with `nbdev` export directives: if given `exported=True`, `filter_msgs_v1` should return **all** exported messages, those marked with SolveIt UI toggle **and** the ones with `nbdev` exported directives.

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    exported:bool=False,
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Filter messages with improved search semantics over `find_msgs`"
    if not exported: return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    if exported: msgs = [m for m in msgs if is_exported(m)]
    return msgs

In [ ]:
ms = await filter_msgs_v1(exported=True)
test_is('_084ce165' in {m.id for m in ms}, True)

In this stage we know four things: one, no argument, empty result; two, thus, unlike `find_msgs_x`, the given arguments will probably be cumulative, add to the result, instead of substract from it; three, the first filter to implement, `exported`; four, next filters? well, we have the example of `filter_msgs_v0`/`exportmsgs` which shows a few more: notes, echo cells, etc. so, we can give ourselves an intermediate goal: refactor the filter logic from `filter_msgs_v0` into `filter_msgs_v1`.

Yes, off the top of my head, one thing come to mind regarding those arguments: instead of listing all possible arguments, let's group them in logical categories: metadata=['exported', ...], msg_type=['code', 'note', ...]

So, `msg_type='note'` would include all notes; `msg_type=['note','code']` would include all note and code messages. `metadata='exported` would include all messages marked as exported; `metadata=['exported','echo']` would include all messages marked as exported or echoed. etc. For now, the semantics of the "'" is **and** ('note','code' really means 'note' and 'code') but perhaps will add additional semantics in the future.

So `msg_type` and `metadata` are the two category axes, each accepting a string or list, with union semantics across and within categories. Let's define the metadata predicate map and rewrite `filter_msgs_v1`:


In [ ]:
#| export
_meta_fns = dict(exported=is_exported, echo=is_echo, header=is_header, hidden=is_hidden, skipped=is_skipped)

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    exclude_metadata:str|list=None,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics across msg_type and metadata flags."
    if not msg_type and not metadata: return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    mt = ([msg_type] if isinstance(msg_type, str) else msg_type) or []
    md = ([metadata] if isinstance(metadata, str) else metadata) or []
    ex = ([exclude_metadata] if isinstance(exclude_metadata, str) else exclude_metadata) or []
    return [m for m in msgs if
            ((mt and m.msg_type in mt) or (md and any(_meta_fns[k](m) for k in md)))
            and not (ex and any(_meta_fns[k](m) for k in ex))]

In [ ]:
ms = await filter_msgs_v1(metadata='exported')
test_is('_084ce165' in {m.id for m in ms}, True)

Does this version replicate `filter_msgs_v0`?

In [ ]:
v0 = await filter_msgs_v0(test_msgs)
v1 = await filter_msgs_v1(test_msgs, metadata=['exported', 'echo', 'header'], exclude_metadata='hidden')
test_eq(v0, v1)

Let's see more examples and edge cases. (Whenever possible, I much prefer to use `test_*` for testing and, sometimes, it's good to display the result, mainly for my own benefit.)

In [ ]:
ms = await filter_msgs_v1()
test_eq(ms, [])

ms = await filter_msgs_v1(test_msgs, msg_type='note')
test_eq(len(ms), 4)
test_is(all(m.msg_type == 'note' for m in ms), True)

ms = await filter_msgs_v1(test_msgs, msg_type=['note', 'prompt'])
test_eq(len(ms), 6)
test_is(all(m.msg_type in ('note', 'prompt') for m in ms), True)

ms = await filter_msgs_v1(test_msgs, metadata='echo')
test_eq(len(ms), 1)
test_eq(ms[0].id, '_fde18916')

ms = await filter_msgs_v1(test_msgs, metadata='exported', exclude_metadata='hidden')
test_is('_89ee9e7c' in {m.id for m in ms}, False)  # exported+hidden excluded
test_is('_8ca984e4' not in {m.id for m in ms}, True) # exporti but also hidden
test_eq(len(ms), 7)

ms = await filter_msgs_v1(test_msgs, msg_type='note', metadata='exported')
test_is('_cdfea4f6' in {m.id for m in ms}, True)  # non-header note, included via exported
test_is('_b6147aa6' in {m.id for m in ms}, True)  # header note, included via both
test_eq(len(ms), 10)

Next: one of the most important thing `filter_msgs_v1` must do is filter by sections. Thus, we need to add section semantics.

Compose with union, like bare ids list. Precedence: ids -> sections -> metadata. Though seldom agents will use ids and sections in the same call, I think.

The heading is included (but can be filtered afterwards by metadata).

Don't see the point of filtering sections by level (levels don't share semantics) and by ID it's cumbersome for humans (but not for LLMs). Let's start with section name: "## name 1", "### name 2" and simple regex.

In [ ]:
#| export
async def get_section(heading:str, msgs:list[MsgX]=None, dname='') -> list[MsgX]:
    "Return messages in section whose header matches heading"
    if not msgs: return await find_msgs_x(header_section=heading, include_skipped=True, dname=dname)
    lvl = len(re.match(r'^(#+)', heading)[1])
    for i,m in enumerate(msgs):
        if m.msg_type != 'note' or not m.content.startswith(heading): continue
        res = [m]
        for m2 in msgs[i+1:]:
            hs = re.match(r'^(#+)', m2.content) if m2.msg_type == 'note' else None
            if hs and len(hs[1]) <= lvl: break
            res.append(m2)
        return res
    return []

In [ ]:
# exact match
sec = await get_section('##### Header (exported)', test_msgs)
test_eq(len(sec), 1)
test_eq(sec[0].content, '##### Header (exported)')

# section spans until next same-level header
sec = await get_section('## message utils', dlog_msgs)
test_is(sec[0].content.startswith('## message utils'), True)
test_is(all(not (m.msg_type=='note' and re.match(r'^#{1,2}[^#]', m.content)) for m in sec[1:]), True)

# no match returns empty
sec = await get_section('## nonexistent section', dlog_msgs)
test_eq(sec, [])

# subsection doesn't bleed into sibling
sec = await get_section('### message tag', dlog_msgs)
last = sec[-1]
test_is(sec[0].content.startswith('### message tag'), True)
test_is(not any(m.content.startswith('### message metadata') for m in sec), True)

len(sec)

2

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    exclude_metadata:str|list=None,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics; result in dialog order."
    if not any([ids, section, msg_type, metadata]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    id_set = set([ids] if isinstance(ids, str) else ids or [])
    secs = ([section] if isinstance(section, str) else section) or []
    sec_ids = {m.id for s in secs for m in await get_section(s, msgs)}
    mt = ([msg_type] if isinstance(msg_type, str) else msg_type) or []
    md = ([metadata] if isinstance(metadata, str) else metadata) or []
    ex = ([exclude_metadata] if isinstance(exclude_metadata, str) else exclude_metadata) or []
    return [m for m in msgs if
            (m.id in id_set or m.id in sec_ids or (mt and m.msg_type in mt) or (md and any(_meta_fns[k](m) for k in md)))
            and not (ex and any(_meta_fns[k](m) for k in ex))]

In [ ]:
# ids: single string
ms = await filter_msgs_v1(test_msgs, ids='_b6147aa6')
test_eq(len(ms), 1)
test_eq(ms[0].id, '_b6147aa6')

# ids: list
ms = await filter_msgs_v1(test_msgs, ids=['_b6147aa6', '_4e4b6f65'])
test_eq(len(ms), 2)

# section: single
ms = await filter_msgs_v1(dlog_msgs, section='### message tag')
test_eq(len(ms), 2)
test_is(ms[0].content.startswith('### message tag'), True)

# section: union with msg_type
ms_sec = await filter_msgs_v1(dlog_msgs, section='### message tag')
ms_union = await filter_msgs_v1(dlog_msgs, section='### message tag', msg_type='note')
test_is(len(ms_union) >= len(ms_sec), True)

# section + exclude_metadata
ms = await filter_msgs_v1(dlog_msgs, section='### message metadata & directives', exclude_metadata='hidden')
test_is(all(not is_hidden(m) for m in ms), True)

# ids + section union (no overlap double-counting)
ms = await filter_msgs_v1(dlog_msgs, ids=dlog_msgs[0].id, section='### message tag')
test_is(dlog_msgs[0].id in {m.id for m in ms}, True)
test_is(ms[0].id == dlog_msgs[0].id, True)  # dialog order preserved

# unknown id: silently absent
ms = await filter_msgs_v1(test_msgs, ids='_nonexistent')
test_eq(ms, [])

Seems to work. Next? (We are progressively approaching relational semantics and probably will end up with our own SQL DSL-like lang/algebra, not for now though.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at what's still missing for practical use, a few natural next steps come to mind:

1. **Range** — `after='_id'` / `before='_id'` or `between=('_id1', '_id2')`: slice a contiguous span by id. Useful for "everything after this header up to that one" without needing `get_section`.

2. **Content search** — `contains='pattern'` (regex): filter by message content. Simple but powerful for finding all cells referencing a symbol, for instance.

3. **`n` / `last`** — `n=10` / `last=5`: take first/last N from the filtered result. Useful for "give me the last 5 exported cells".

Of these, **range** feels most complementary to what's already there and closest to what `get_section` already does implicitly. Content search would open up a new axis entirely. `n`/`last` is more of a post-filter convenience.

Which direction interests you most?

<details class='token-usage-details'><summary>$0.0401</summary>

`total=35,673 | in=35,006 | out=667 | cached=94.4% | cache_new=1,251 | searches=0 | $0.0401`

</details>


`range`, `last` and any operation picking messages by id, I think would be better served by separate helpers whose results can be  passed directly with the `ids` argument to filter additionally by metadata. Don't you think so?

More akin to what we're doing with `filter_msgs_v1` would be introducing "not" semantics with (say) "!" operator. For example, "!exported" would mean "not exported", "!code" -> ['note', 'prompt`, 'raw'], or "!### a section" would mean "everything but section ### a section". Going to DSL step by step.

So the full updated semantics are:

- Parse `!`-prefixed terms from each axis as negations, the rest as positives
- Union all positives across all axes; if none, start from full `msgs`
- Subtract globally all negation matches
- `exclude_metadata` goes away, replaced by e.g. `metadata='!hidden'`

In [ ]:
#| export
def _parse_terms(v):
    terms = ([v] if isinstance(v, str) else v) or []
    return [t for t in terms if not t.startswith('!')], [t[1:] for t in terms if t.startswith('!')]

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pos = set(ids_l)
    for s in sec_pos: pos |= {m.id for m in await get_section(s, msgs)}
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    neg = set()
    for s in sec_neg: neg |= {m.id for m in await get_section(s, msgs)}
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    base = pos if pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

Tests in SolveIt (fastai style) serve two purposes: correctnes **and** documentation. I think it'd be better to implemente minimal synthetic tests with a minimal set of synthetic (very simple dicts/MsgX) so we can see results without going to other dialog lists elsewhere. For instance, for negation only, we can simply create two minimal MsgX and test visually. And so on. At the end we can go back to the test set and even this full dialog.

In [ ]:
def mk(id, content='',  msg_type='note', **kwargs):
    (res := AD(id=id, msg_type=msg_type, content=content, is_exported=0, output='', skipped=0, pinned=0)).update(**kwargs)
    return msgx(res)

In [ ]:
# negation only: all msgs minus hidden
syn = mk('a', '#| hide', 'code'), mk('b')
ms = await filter_msgs_v1(syn, metadata='!hidden')
test_eq(ms, [syn[1]])

# positive + negation: exported but not hidden
syn = mk('a', '#| export\nx=1', 'code'), mk('b', '#| hide\n#| export\ny=2', 'code'), mk('c', 'just a note')
ms = await filter_msgs_v1(list(syn), metadata=['exported', '!hidden'])
test_eq(ms, [syn[0]])

# msg_type negation: everything but code
syn = mk('a', '## h'), mk('b', 'x=1', 'code'), mk('c', 'q', 'prompt')
ms = await filter_msgs_v1(list(syn), msg_type='!code')
test_eq(ms, [syn[0], syn[2]])

# section negation: all minus a section
syn = mk('a', '## sec1'), mk('b', 'x=1', 'code'), mk('c', '## sec2'), mk('d', 'y=2', 'code')
ms = await filter_msgs_v1(list(syn), section='!## sec1')
test_eq(ms, [syn[2], syn[3]])

# combined: code msgs minus those in a section
ms = await filter_msgs_v1(list(syn), msg_type='code', section='!## sec1')
test_eq(ms, [syn[3]])

# all-negative: full set minus code and hidden
syn = mk('a', '## h'), mk('b', '#| hide\nx=1', 'code'), mk('c', 'y=2', 'code'), mk('d', 'q', 'prompt')
ms = await filter_msgs_v1(list(syn), msg_type='!code', metadata='!hidden')
test_eq(ms, [syn[0], syn[3]])

In [ ]:
v0 = await filter_msgs_v0(test_msgs)
v1 = await filter_msgs_v1(test_msgs, metadata=['exported', 'echo', 'header', '!hidden'])
test_eq(v0, v1)

### content search
Regex filter on message content. Simple, composable, immediately useful for "find all cells referencing symbol foo".

`contains` fits naturally as another axis in `filter_msgs_v1`, with `!` negation like the rest. 

We can use the exact same args than `find_msgs`:
```python
async def find_msgs(
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used)
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
```
Just one pattern (with `or` logic if you want multiple matches)

In [ ]:
#| export
def _content_match(m, pat, use_case, use_regex):
    flags = (re.DOTALL|re.MULTILINE)|(0 if use_case else re.IGNORECASE)
    return bool(re.search(pat, m.content, flags) if use_regex else (pat in m.content if use_case else pat.lower() in m.content.lower()))

In [ ]:
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    re_pattern:str='',
    use_case:bool=False,
    use_regex:bool=True,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([ids_l, sec_pos, mt_pos, md_pos, (pat_pos and not pat_neg)])
    pos = set(ids_l)
    for s in sec_pos: pos |= {m.id for m in await get_section(s, msgs)}
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    for s in sec_neg: neg |= {m.id for m in await get_section(s, msgs)}
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

In [ ]:
syn = [mk('a', 'x = foo + 1', 'code'), mk('b', 'y = bar + 2', 'code'), mk('c', 'foo is great', 'note')]

ms = await filter_msgs_v1(list(syn), re_pattern='foo')
test_eq([m.id for m in ms], ['a', 'c'])

ms = await filter_msgs_v1(list(syn), re_pattern='!foo')
test_eq([m.id for m in ms], ['b'])

ms = await filter_msgs_v1(list(syn), re_pattern='FOO', use_case=True)
test_eq(ms, [])

ms = await filter_msgs_v1(list(syn), re_pattern='FOO', use_case=False)
test_eq([m.id for m in ms], ['a', 'c'])

ms = await filter_msgs_v1(list(syn), re_pattern='foo|bar')
test_eq([m.id for m in ms], ['a', 'b', 'c'])

### section map
Builds a lookup from each markdown section header to the messages that belong to that section, including nested subsections. It lets us treat a dialog like a structured document and ask, “which messages are in this section?”

In [ ]:
#| export
def _header_level(m):
    if m.msg_type != 'note' or not m.content.startswith('#'): return 0
    return len(re.match(r'^(#+)', m.content)[1])


In [ ]:
test_eq(_header_level(mk('a', '### foo')), 3)

In [ ]:
#| export
def section_map(msgs):
    "Return dict mapping full header line -> list of msg ids in that section (inclusive of header)"
    res, cur_hdrs = {}, []
    for m in msgs:
        lvl = _header_level(m)
        if lvl:
            cur_hdrs = [(l,h) for l,h in cur_hdrs if l < lvl]
            hdr = re.match(r'^(#+[^\n]*)', m.content)[1]
            cur_hdrs.append((lvl, hdr))
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
        else:
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
    return res

In [ ]:
def mk_note(id, content): return mk(id, content)
def mk_code(id, content='x=1'): return mk(id, content)

test_eq(section_map([]), {})
test_eq(section_map([mk_code('a'), mk_code('b')]), {})

# messages before first header are not in any section
msgs = [mk_code('pre'),  mk_note('h1', '## Top'), mk_code('c1')]
sm = section_map(msgs)
test_is('pre' not in str(sm), True)
test_eq(sm['## Top'], ['h1', 'c1'])

# nested: child msgs belong to both parent and child sections
msgs = [ mk_note('h1','## Top'),  mk_note('h2','### Sub'), mk_code('c1')]
sm = section_map(msgs)
test_eq(sm['## Top'], ['h1', 'h2', 'c1'])
test_eq(sm['### Sub'], ['h2', 'c1'])

# sibling headers don't bleed into each other
msgs = [ mk_note('h1','## A'), mk_code('c1'),  mk_note('h2','## B'), mk_code('c2')]
sm = section_map(msgs)
test_eq(sm['## A'], ['h1', 'c1'])
test_eq(sm['## B'], ['h2', 'c2'])

# header with no children
test_eq(section_map([ mk_note('h1','## Lonely')]), {'## Lonely': ['h1']})

sm = section_map(await find_msgs(include_meta=True))
sm.keys()

dict_keys(['# dialog', '## message utils', '### message tag', '### message metadata & directives', '### notes disclosure', '### message extended', '### extended `find_msgs`', '## dialog utils', '### dialog path', '### dialog module', '### dialog export', '### dialog link', '### empty .ipynb', '### message id', '### get output', '### output formatting', '### markdown splitter', '### token count', '## Code message disclosure', '### disclose code', '### dialog and notebook disclosure', '### tests', '### Code disclosure: rationale, decisions, current state', '## distill', '### filter messages v0', '### digest dialog', '### filter messages v1', '### content search', '### section map', '### filter messages v1 wrap up', '### filter messages v2', '### handoff test', '## token budget', '## next', '## branching', '## dependency tracking\\isolation', '## import', '## export -'])

In [ ]:
sm = section_map(dlog_msgs)
list(sm.keys())

['# dialog',
 '## message utils',
 '### message tag',
 '### message metadata & directives',
 '### notes disclosure',
 '### message extended',
 '### extended `find_msgs`',
 '## dialog utils',
 '### dialog path',
 '### dialog module',
 '### dialog export',
 '### dialog link',
 '### empty .ipynb',
 '### message id',
 '### get output',
 '### output formatting',
 '### markdown splitter',
 '### token count',
 '## Code message disclosure',
 '### disclose code',
 '### dialog and notebook disclosure',
 '### tests',
 '### Code disclosure: rationale, decisions, current state',
 '## distill',
 '### filter messages v0',
 '### digest dialog',
 '### filter messages v1',
 '### content search',
 '### section map',
 '### section regex',
 '### filter messages v1 wrap up',
 '### filter messages v2',
 '### handoff test',
 '## token budget',
 '## next',
 '## branching',
 '## dependency tracking\\isolation',
 '## import']

Are all sections detected correctly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

For the current dialog, **yes, it looks correct for standard left-aligned ATX headers**:

- `#`, `##`, `###`, etc. at the start of a note are detected
- nesting is handled the way you’d expect
- sibling sections don’t bleed into each other

But **not all markdown sections in general** are detected:

- headings with leading spaces before `#` are missed
- setext headings like `Title\n---` are missed
- anything that isn’t a note starting exactly with `#` is ignored

So the short answer is:

- **correct for the conventions you’re using here**
- **not fully general markdown section detection**

<details class='token-usage-details' markdown='1'><summary>$0.0489</summary>

`total=39,353 | in=38,177 | out=1,176 | cached=5.0% | reasoning=1,034 | searches=0 | $0.0489`

</details>


In Solveit, sections are note message whose first line start with '#', is a convention enforced by `find_msgs`. What are "standard left-aligned ATX headers"? What are "setext headings"? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In Markdown, those are two common header styles:

- **ATX headers**: headings written with leading `#` characters
- **Setext headings**: headings written as a line of text followed by an underline of `=` or `-`

“**Standard left-aligned ATX headers**” means headings like this, with the `#` at the start of the line and no leading spaces:

```markdown
# Title
## Section
### Subsection
```

Examples that are **not** standard left-aligned ATX in your convention:

```markdown
  # indented heading
```

or

```markdown
text # not a heading
```

A **setext heading** looks like this:

```markdown
Title
=====
```

or

```markdown
Section
-------
```

In Markdown:

- `=` means a level-1 heading
- `-` means a level-2 heading

So in your Solveit convention, section detection only uses the first style: note messages whose first line starts with `#`.

<details class='token-usage-details' markdown='1'><summary>$0.0435</summary>

`total=38,754 | in=38,423 | out=331 | cached=5.0% | reasoning=115 | searches=0 | $0.0435`

</details>


### section regex

```python
def _scoped_msgs(msgs, ids, section):
    scoped = set()
    if ids is not None: scoped |= ({ids} if isinstance(ids, str) else set(ids))
    if section:
        secs = [section] if isinstance(section, str) else list(section)
        sm = section_map(msgs)
        for s in secs:
            for k,v in sm.items():
                if re.fullmatch(s, k, re.IGNORECASE): scoped |= set(v)
    return [m for m in msgs if m.id in scoped]

async def filter_msgs(msg_type=None, metadata=None, ids=None, section=None, dname='', msgs=None) -> list[Msg]:
    "Filter messages with two-phase semantics: scope (ids, section) then union-filter (msg_type, metadata)"
    if not any([msg_type, metadata, ids is not None, section]): return []
    if msgs is None: msgs = await find_msgs(include_skipped=True, dname=dname)
    if ids is not None or section: msgs = _scoped_msgs(msgs, ids, section)
    if not any([msg_type, metadata]): return msgs
    typs = {msg_type} if isinstance(msg_type, str) else set(msg_type or [])
    metas = {metadata} if isinstance(metadata, str) else set(metadata or [])
    seen, res = set(), []
    for m in msgs:
        if m.id in seen: continue
        if (typs and m.msg_type in typs) or (metas and any(_meta_fns[k](m) for k in metas)):
            res.append(m); seen.add(m.id)
    return res
```

See above an old version of `filter_msgs` that is mostly superseded by out `filter_msgs_v1`. Nevertheless it had a good feature, regex for section titles. Woudl it be hard to add to our version?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Not hard at all. Currently `get_section` matches via `m.content.startswith(heading)` — exact prefix. We'd replace those calls in `filter_msgs_v1` with a `section_map` lookup using `re.fullmatch`, exactly like `_scoped_msgs` does.

The only small design question: `fullmatch` (whole key must match the pattern) or `search` (pattern anywhere in key)? `fullmatch` seems safer to avoid unintended matches.

<details class='token-usage-details'><summary>$0.2859</summary>

`total=50,059 | in=49,744 | out=315 | cached=0.0% | cache_new=48,871 | searches=0 | $0.2859`

</details>


No vey usefull though, maybe later.

### filter messages v1 wrap up
`filter_msgs_v1` is the dialog query helper for selecting messages by ids, sections, message type, metadata, and content, with support for negation and dialog-order results.

Let's wrap up this section with a modularized version of `filter_msgs_v1` with `scoped_msgs` as discussed above and a new `export_msgs` if necessary.

**modularize.** Extract the scope phase into `_scoped_msgs(msgs, ids, section)` (already sketched in the old `filter_msgs`), leaving `filter_msgs_v1` with just the union/negation filter logic. No behavior change, just cleaner structure.

`_scoped_msgs` first.

The scope phase in `filter_msgs_v1` currently handles `ids` and section pattern resolution (both pos and neg). A clean extraction:

In [ ]:
#| export
def _resolve_sections(msgs, patterns):
    "Return set of msg ids matching any of the section patterns (supports regex via section_map)"
    sm = section_map(msgs)
    ids = set()
    for p in patterns:
        for k, v in sm.items():
            if re.fullmatch(p, k, re.IGNORECASE): ids |= set(v)
    return ids

This replaces the `get_section` calls in `filter_msgs_v1`, and also upgrades section matching to use `section_map` + regex (like the old `filter_msgs`). Then `filter_msgs_v1` calls it for both `sec_pos` and `sec_neg`.

In [ ]:
#| export
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    re_pattern:str='',
    use_case:bool=False,
    use_regex:bool=True,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([ids_l, sec_pos, mt_pos, md_pos, (pat_pos and not pat_neg)])
    pos = set(ids_l)
    pos |= _resolve_sections(msgs, sec_pos)
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    neg |= _resolve_sections(msgs, sec_neg)
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

In [ ]:
this_idx = await msg_idx()
msgs = await find_msgs_x()
len(msgs)

231

In [ ]:
v1 = await filter_msgs_v1(msgs[:this_idx], section=['## message utils', '## dialog utils'], metadata=['exported', 'echo', 'header', '!hidden'])
len(v1)

132

It seems a lot of messages. Let's check this is correct. All messages in that list should comply with the constraints: be a header or exported and not hidden, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right. Let's verify both constraints — section membership and the metadata predicate:

In [ ]:
sec_ids = _resolve_sections(msgs[:this_idx], ['## message utils', '## dialog utils'])
if all(is_header(m) or (is_exported(m) and not is_hidden(m)) or is_echo(m) for m in v1) is False:
    print("metadata constraints FAIL")
if all(m.id in sec_ids for m in v1) is False: print('Sections FAIL')

metadata constraints FAIL
Sections FAIL


Some messages in `v1` are not in `sec_ids`. This reveals a design ambiguity: `section` in `filter_msgs_v1` contributes to the *union* (adds section members to the result), but the call was intended to mean "within those sections, return exported/header/echo messages" — i.e. *intersection*.

The old `filter_msgs` handled this with two explicit phases: scope first, then filter. `filter_msgs_v1` doesn't have that distinction.

Two options:
1. Add a `scope` parameter separate from `section` for intersection semantics
2. Keep `section` as union but document it clearly, and use a separate call pattern (pass pre-scoped `msgs`) for intersection

I think the old `filter_msgs` was more intuitive than our version: first scoped the target set, then filter out the result. In fact, I don't understand our version at all, have to spent much more time studying it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a fair observation. The two-phase mental model is much clearer:

> *"Give me messages **in** section X (or with ids Y), **that are also** exported/header/echo."*

vs. `filter_msgs_v1`'s union model which reads more like:

> *"Give me messages that are **any of**: in section X, exported, header, echo."*

The union model is powerful for additive queries but breaks down for "filter within a scope" — which is probably the more common real-world need.

Would you like to redesign `filter_msgs_v1` around the two-phase approach, or keep it as-is and introduce a separate function for scoped filtering?

<details class='token-usage-details'><summary>$0.0440</summary>

`total=53,699 | in=53,304 | out=395 | cached=95.9% | cache_new=1,949 | searches=0 | $0.0440`

</details>


### filter messages v2
Query helper for selecting dialog messages by id or section, then filtering by message type, metadata, or content pattern. Supports negation and preserves dialog order.

Let's do a `filter_msgs_v2`. Worth keeping the `v1` for now (need to study it). Try to make it as modular as possible delegating out to external helpers that we already have or new.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Before writing code, let's agree on the shape. Two clean phases:

**Phase 1 — scope** (`_scope_msgs`): if `ids` or `section` given, narrow the working set. If neither given, full `msgs` is the scope.

**Phase 2 — filter** (`_filter_pos_neg`): if `msg_type`/`metadata`/`re_pattern` given, apply union+negation within the scoped set. If none given, return the scope as-is.

Helpers already available: `_resolve_sections`, `_parse_terms`, `_meta_fns`, `_content_match`. The only new ones would be `_scope_msgs` and `_filter_pos_neg`.

Does that decomposition match your mental model?

<details class='token-usage-details'><summary>$0.0375</summary>

`total=54,056 | in=53,570 | out=486 | cached=99.0% | cache_new=270 | searches=0 | $0.0375`

</details>


It does. But your opinion is as important as mine. In fact you have in your weights a lot more info than I could possibly ever have, have seen lot of similar cases and know what's the intended use case of this helpers. What's your take?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

My take: the two-phase design is right, and `filter_msgs_v1`'s union model is worth keeping as a lower-level primitive — it's essentially set algebra on message ids, useful when you want to *compose* result sets. `filter_msgs_v2` is the higher-level, more ergonomic API that most callers will actually use.

One thing I'd flag: `_resolve_sections` is currently `async` but doesn't need to be — `section_map` is sync. Making it sync simplifies the call chain and lets `_scope_msgs` also be sync, so only the top-level `filter_msgs_v2` needs to be async (for the `find_msgs_x` fetch). Worth fixing before we build on it.

<details class='token-usage-details'><summary>$0.0371</summary>

`total=54,321 | in=53,860 | out=461 | cached=99.0% | cache_new=283 | searches=0 | $0.0371`

</details>


Fair enough, go on.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

First, make `_resolve_sections` sync:

Then `_scope_msgs`:

In [ ]:
#| export
def _scope_msgs(msgs, ids, section):
    "Narrow msgs to those matching ids and/or section patterns."
    if ids is None and not section: return msgs
    scoped = set()
    if ids is not None: scoped |= {ids} if isinstance(ids, str) else set(ids)
    if section:
        secs = [section] if isinstance(section, str) else section
        scoped |= _resolve_sections(msgs, secs)
    return [m for m in msgs if m.id in scoped]

In [ ]:
this_idx = await msg_idx()
msgs = (await find_msgs_x(include_skipped=True))[:this_idx]

section = ['## message utils', '## dialog utils', '## Code message disclosure', '## distill']
scpd = _scope_msgs(msgs, None, section)
test_eq('def _scope_msgs(msgs, ids, section):' in scpd[-1].content, True)
len(scpd)

201

`_filter_pos_neg` — applies union+negation within an already-scoped list:

In [ ]:
#| export
def _filter_pos_neg(msgs, msg_type=None, metadata=None, re_pattern='', use_case=False, use_regex=True):
    "Apply union+negation filter to msgs. If no positive terms, return msgs as-is."
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([mt_pos, md_pos, pat_pos and not pat_neg])
    if not has_pos and not any([mt_neg, md_neg, pat_neg]): return msgs
    pos = set()
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]

In [ ]:
fltd = _filter_pos_neg(scpd, metadata=['exported', 'echo', 'header', '!hidden'])
test_eq('def _scope_msgs(msgs, ids, section):' in fltd[-1].content, True)
len(fltd)

82

As before, I sincerely don't understand `_filter_pos_neg`. I don't say it's wrong, simply I don't get it. Shouldn't each step/filter narrow down the target `msgs` progressively. I see the function iterating every time in the same set of messages and building IMHO an unnecessary set of ids.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The tension is that progressive narrowing gives **AND** semantics ("exported AND header"), but the design calls for **OR** among positives ("exported OR header OR echo"). You can't narrow progressively for OR — you need to collect then union.

That said — is OR actually what you want among positive terms? Or would AND be more natural after scoping?

<details class='token-usage-details'><summary>$0.0410</summary>

`total=55,946 | in=55,519 | out=427 | cached=97.7% | cache_new=1,011 | searches=0 | $0.0410`

</details>


You're right again. `OR` is the correct semantics, I have to refresh my logic. Let's test this.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed, the `v1` signature is cleaner — the visual grouping of scope vs filter args makes the two-phase semantics self-documenting. So `filter_msgs_v2` keeps that exact signature, just changes the internal logic to two-phase (scope then OR-filter) instead of pure union. Ready to write it?

<details class='token-usage-details'><summary>$0.3160</summary>

`total=56,035 | in=55,958 | out=77 | cached=0.0% | cache_new=55,481 | searches=0 | $0.3160`

</details>


In [ ]:
#| export
async def filter_msgs_v2(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    ids:str|list=None, # Message ids to include (union with section).
    section:str|list=None, # Section patterns to include (union with ids).
    msg_type:str|list=None, # Message types to include.
    metadata:str|list=None, # Metadata keys to to filter on ('exported', 'echo', 'header', 'hidden').
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used).
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Two-phase filter: scope by ids/section, then OR-filter by type/metadata/pattern with ! negation."
    if not any([ids is not None, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return _filter_pos_neg(_scope_msgs(msgs, ids, section), msg_type, metadata, re_pattern, use_case, use_regex)

In [ ]:
#| echo
syn = [mk('a', '## visible'), mk('b', '## skipped', skipped=1), mk('c', 'x=1', 'code')]
ms = await filter_msgs_v2(syn, metadata=['header', '!skipped'])
test_eq(ms, [syn[0]])

In [ ]:
this_idx = await msg_idx()

### handoff test

Generate a markdown that will serve us as a hand-off to jump start a new discussion easily with the adecuate context. We'll discuss together if that markdown plus additional references and prompts is adecuate for the future task.

Capture: current API surface (`filter_msgs_v2`, `get_section`, `section_map`, `exportmsgs`), the `clamp_msg` design table, the two-axis `note`/`code` levels, the skeleton rules, and pointers to references to check before writing starting a new session.

In [ ]:
msgs = (await find_msgs_x(include_skipped=True))[:this_idx]

v2 = await filter_msgs_v2(msgs, section=['## message utils', '## dialog utils', '## Code message disclosure', '## distill'], metadata=['exported', 'echo', 'header', '!hidden', '!skipped'])

In [ ]:
sm = section_map(v2)
list(sm.keys())

['## message utils',
 '### message tag',
 '### message metadata & directives',
 '### notes disclosure',
 '### message extended',
 '### extended `find_msgs`',
 '## dialog utils',
 '### dialog path',
 '### dialog module',
 '### dialog export',
 '### dialog link',
 '### empty .ipynb',
 '### message id',
 '### get output',
 '### output formatting',
 '### markdown splitter',
 '### token count',
 '## Code message disclosure',
 '### disclose code',
 '### dialog and notebook disclosure',
 '### tests',
 '### Code disclosure: rationale, decisions, current state',
 '## distill',
 '### filter messages v0',
 '### digest dialog',
 '### filter messages v1',
 '### content search',
 '### section map',
 '### filter messages v1 wrap up',
 '### filter messages v2']

In [ ]:
fmt = 'md'
md = exportmsgs(v2, hdr_clamp='header++', fmt=fmt,
    frontmatter=dict(path=find_dname(), description='filter messages sections of pote.dutil dialog', n_msgs=len(msgs), fmt=fmt))
# print(md)
await link_msg(md);

---
digest: solveit-dialog
path: /prj/pote/nbs/01_dialog
exported_at: 2026-05-18T09:33:17+00:00
description: 'filter messages sections of pote.dutil dialog'
stats: {total: 213, exported: 84}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_e069f418 note -->
## message utils
Core primitives for working with solveit messages: a `Msg` type alias, utilities for reading nbdev directives and message metadata, progressive disclosure of note content, and an extended `MsgX` subclass with convenience properties.

<!-- #_ea3120ed note -->
### message tag
Defines `Msg` as a type alias for `adict`, providing a concise, readable type annotation for solveit message objects throughout the dialog.

<!-- #_e2f454e8 code -->
```python
Msg: TypeAlias = AD
```

<!-- #_7d419edc note -->
### message metadata & directives
Utilities for inspecting solveit message metadata. `is_exported` checks if a message is marked for export (via solveit metadata or nbdev `#|export`/`#|exports`/`#|exporti` directives). `is_hidden` checks if a message is hidden from doc generation via `#|hide`. `is_echo` checks if a code message is marked to show source via `#|echo`. `is_header` checks if the note message start with a markdown heading.

<!-- #_084ce165 code -->
```python
_exports = {'exporti', 'exports', 'export'}
_hidden = {'hide'}
_echo = {'echo', 'echo:'}
```

<!-- #_8e71d4cb code -->
```python
def is_exported(m:Msg) -> bool:
    "Is message exported with solveit metadata or nbdev directive?"
    return m.is_exported or (m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _exports))

def is_hidden(m:Msg) -> bool:
    "Is message hidden by a nbdev directive (only for doc generation, not AI visibility)?"
    return m.msg_type == 'code' and bool(
        extract_directives(AD(source=m.content), remove=False).keys() & _hidden)

def is_echo(m:Msg) -> bool:
    "Does message have an echo directive that isn't false?"
    if m.msg_type != 'code': return False
    d = {k:v for k,v in extract_directives(AD(source=m.content), remove=False).items() if k in _echo}
    return bool(d) and all('false' not in v for v in d.values())

def is_header(m:Msg) -> bool:
    "Does note message content start by a markdown header?"
    return m.msg_type == 'note' and m.content.startswith('#')

def is_skipped(m:Msg) -> bool: return bool(m.skipped)
```

<!-- #_c35ba3c6 code -->
```python
test_is(is_hidden(Msg(msg_type='code', content='#|hide\n\nsome content')), True)

test_is(is_header(Msg(msg_type='note', content='# a')), True)
test_is(is_header(Msg(msg_type='note', content=' ### a')), False)

test_is(is_echo(Msg(msg_type='code', content='#| echo\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#|hide\n#| echo: true\nprint(1)')), True)
test_is(is_echo(Msg(msg_type='code', content='#| echo: false\nprint(1)')), False)
test_is(is_echo(Msg(msg_type='note', content='#| echo\nsome text')), False)
test_is(is_echo(Msg(msg_type='code', content='print(1)')), False)
```

<!-- #_08e4b3f3 note -->
### notes disclosure
`clamp_note` supports four levels of **progressive disclosure** for markdown notes.
| Level | Returns |
|---|---|
| `'header'` | The first line, if it is a markdown header; otherwise `''` |
| `'header+'` | Header + first non-blank line after it |
| `'header++'` | Header + first paragraph, only if it immediately follows the header |
| `int` (e.g. `30`) | First *n* lines, regardless of structure |
All levels return `''` if the content is empty or (for header levels) has no leading header.

<!-- #_c087a3d1 code -->
```python
def clamp_note(
    c:str, # Markdown note content
    level:str|int='header' # Disclosure level: 'header', 'header+', 'header++', or int for max lines
) -> str:
    "Return a progressively disclosed excerpt of a markdown note."
    c = c.strip().splitlines()
    if c:
        if isinstance(level, int): return '\n'.join(c[:level])
        if level == 'header': return c[0] if c[0][0] == '#' else ''
        if level == 'header+':
            i, res = 0, c[0]
            if res[0] != '#': return ''
            while (i:=i+1) < len(c):
                if c[i]: return f"{res}\n{c[i]}"
            return res
        if level.startswith('header++'):
            header = c[0] if c[0][0] == '#' else ''
            if not header: return ''
            blocks = [b.strip() for b in '\n'.join(c[1:]).split('\n\n') if b.strip()]
            if blocks:
                first = blocks[0].splitlines()[0]
                if not first.startswith('#') and not first.startswith(('```', '~~~')):
                    return f"{header}\n{blocks[0]}"
            return header
    return ''
```

<!-- #_cd7c3a8e code -->
```python
cnt = '#### test\n> Paragraph 1.\nLine.\n\nParagraph 2\n```python\na=13\n```'
test_eq(clamp_note(cnt, 'header'), '#### test')

test_eq(clamp_note(cnt, 'header+'), '#### test\n> Paragraph 1.')
test_eq(clamp_note(cnt, 'header++'), '#### test\n> Paragraph 1.\nLine.')
```

<!-- #_45ef157a note -->
### message extended
Extends the base `Msg` (`adict`) type with `MsgX`, a subclass that adds convenience properties for inspecting message metadata.
`msgx()` casts any `Msg` in-place to `MsgX` (zero-copy, via `__class__` reassignment). `MsgX` then exposes `.exported_x` and `.hidden_x` as properties wrapping `is_exported`/`is_hidden`.

<!-- #_04219f9f code -->
```python
class MsgX(Msg): ...
```

<!-- #_fc5b3723 code -->
```python
def msgx(msg:adict) -> None:
    "Return `msg` cast to MsgX."
    msg.__class__ = MsgX
    return msg
```

<!-- #_dcb3edce code -->
```python
@patch(as_prop=True)
def exported_x(self:MsgX) -> bool: return is_exported(self)
@patch(as_prop=True)
def hidden_x(self:MsgX) -> bool: return is_hidden(self)
@patch(as_prop=True)
def echo_x(self:MsgX) -> bool: return is_echo(self)
@patch(as_prop=True)
def header_x(self:MsgX) -> bool: return is_header(self)
```

<!-- #_d8cca03f note -->
### extended `find_msgs`
`find_msgs` returning MsgX.

<!-- #_67ed7e1f code -->
```python
@delegates(find_msgs)
async def find_msgs_x(*args, **kwargs) -> list[MsgX]:
    "find_msgs returning MsgX instances."
    ms = await find_msgs(*args, **kwargs)
    for m in ms: m.__class__ = MsgX
    return ms
```

<!-- #_66b148e5 note -->
## dialog utils
Utilities for treating dialogs as first-class objects: resolving their backing files and Python modules, and introspecting runtime state such as cell outputs.

<!-- #_c945a564 note -->
### dialog path
Resolves the filesystem path of a solveit dialog's backing `.ipynb` notebook file.

<!-- #_05549ac3 code -->
```python
def nb_path(dname:str=''): return (Path.home()/(dname or find_dname()).removeprefix('/')).with_suffix('.ipynb')
```

<!-- #_aad8cecd note -->
### dialog module
Resolves the nbdev module name declared in a dialog or solo module if not found, enabling other tools to know which Python module a dialog exports to (if using `dlg_export`).

<!-- #_7419f2b9 code -->
```python
_exp_pat =r'^\s*#\s*\|\s*default_exp\s+(\w+)'
async def dlg_module(dname:str='', solo_nb:bool=False):
    "Finds the nbdev default_exp in a dialog, or the solo module name if not nbdev"
    from nbdev.config import is_nbdev
    if not is_nbdev(p := nb_path(dname)) or solo_nb: return p.stem
    msgs = await find_msgs(_exp_pat, dname=(dname or find_dname()), limit=1, include_output=False, include_meta=False)
    if msgs: return msgs[0].content.strip().split()[-1]
```

<!-- #_0ed44ba4 note -->
### dialog export
Triggers nbdev export for a dialog, compiling its exported cells into the corresponding Python module on disk.

<!-- #_0872c7b7 code -->
```python
async def dlg_export(dname:str='', solo_nb:bool=False):
    "Export dialog `dname` to Python module"
    from nbdev.export import nb_export
    from nbdev.config import get_config, is_nbdev
    p = nb_path(dname)
    if not (_is := is_nbdev(p)): solo_nb = True
    mod = await dlg_module(dname, solo_nb=solo_nb)
    lib_path = p.parent if solo_nb else get_config(p).lib_path if _is else Path('.')
    if solo_nb: nb_export(p, lib_path=lib_path, name=mod, solo_nb=solo_nb)
    else: nb_export(p)
    return (lib_path/f"{mod}.py").exists()
```

<!-- #_148b336f note -->
### dialog link
Generates a clickable HTML hyperlink to any solveit dialog by path, for use in notes and outputs.

<!-- #_d11979a8 code -->
```python
def get_dialog_link(
    path:str # Path to dialog (e.g. '/aai-ws/dialoghelper/nbs/00_core')
) -> str:
    "Return an HTML link to open a dialog in Solveit."
    from dialoghelper.core import dialog_link
    return dialog_link(path).data
```

<!-- #_e492e9bc note -->
### empty .ipynb
Empty .ipynb file.

<!-- #_e7525b46 code -->
```python
def empty_dialog_nb() -> str:
    "Return a minimal dialog notebook json"
    return '''
{
"cells": [],
"metadata": {
"solveit_dialog_mode": "learning",
"solveit_ver": 2
},
"nbformat": 4,
"nbformat_minor": 5
}
'''
```

<!-- #_4e57fd27 note -->
### message id
Exposes the id of the currently executing message, enabling code to self-reference its own position in the dialog.

<!-- #_17421964 code -->
```python
async def get_msg_id():
    "Get current message id."
    return (await read_msg(0)).id
```

<!-- #_53259154 note -->
### get output
Retrieves the rendered output string of any message by id, enabling programmatic inspection of what a cell produced.

<!-- #_da13ebb8 code -->
```python
@delegates(read_msg)
async def get_output(id:str=None, **kwargs) -> list[str]:
    msg = await read_msg(0, id=id, **kwargs)
    return msg.output
```

<!-- #_7e40c2ac note -->
### output formatting
Converts Python objects into a JSON-encoded display payload — capturing all available MIME representations (plain text, HTML, markdown) — for programmatic use outside the normal cell output pipeline.

<!-- #_88c3710d code -->
```python
_format = get_ipython().display_formatter.format if in_dialog() else None
```

<!-- #_fc332cae code -->
```python
delegates(_format)
def format_output(o, **kwargs):
  d, md = _format(o, **kwargs) if _format else ('', {})
  return json.dumps([{"data": d, "metadata": md, "output_type": "display_data"}])
```

<!-- #_c0442b3d note -->
### markdown splitter
Splits a markdown string into its constituent blocks, separating fenced code blocks from surrounding prose, so callers can process or transform each block type independently.

<!-- #_7013a81a code -->
```python
def split_md_blocks(md):
    "Split markdown into fenced/non-fenced blocks via mistletoe AST"
    with MarkdownRenderer() as renderer:
        doc = Document(md)
        blocks,buf = [],[]
        for c in doc.children:
            if isinstance(c, CodeFence):
                if buf: blocks.append(''.join(buf)); buf = []
                blocks.append(renderer.render(c))
            else: buf.append(renderer.render(c))
        if buf: blocks.append(''.join(buf))
        return blocks
```

<!-- #_ccaf18fa note -->
### token count

<!-- #_c0fa4c65 note -->
## Code message disclosure
Progressive disclosure of Python source code at three levels: signatures only (level 1), signatures with docstrings and comments (level 2), and full source (level 3). Per-cell, preserving message boundaries and ids.

<!-- #_8f869e8c note -->
### disclose code

<!-- #_81a301e3 code -->
```python
def _is_docstring(n): return isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant) and isinstance(n.value.value, str)

def _reindent(src, n): return '\n'.join(' '*n + l for l in src.splitlines())

def _postprocess(s):
    "Collapse triple+ blank lines and hoist lone `...` body onto the signature line"
    s = re.sub(r'\n{3,}', '\n\n', s)
    return re.sub(r':\n[ \t]+\.\.\.', ': ...', s)

_compound_types = (ast.If, ast.For, ast.While, ast.With, ast.Try, ast.Match)
```

<!-- #_2e4f5500 code -->
```python
def _strip_private(tree):
    "Remove private (underscore-prefixed) top-level nodes and class methods from `tree` in-place"
    is_priv = lambda n: getattr(n, 'name', '').startswith('_')
    tree.body = [n for n in tree.body if not is_priv(n)]
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            node.body = [n for n in node.body if not is_priv(n)]
    return tree
```

<!-- #_86befa71 code -->
```python
def stub_fn(atok, fn, level=1):
    "Return stubbed source for function `fn` at disclosure `level`"
    if level == 3: return atok.get_text(fn)
    start_line = fn.decorator_list[0].lineno if fn.decorator_list else fn.lineno
    body_start = fn.body[0].lineno - start_line
    if level == 1 or body_start == 0:
        n = copy.deepcopy(fn)
        n.body = [ast.Expr(ast.Constant(...))]
        return _reindent(ast.unparse(n), fn.col_offset)
    indent = ' ' * fn.body[0].col_offset
    lines = atok.get_text(fn).splitlines()
    if _is_docstring(fn.body[0]):
        doc_end = fn.body[0].end_lineno - start_line + 1
        return '\n'.join(lines[:body_start] + lines[body_start:doc_end] + [f'{indent}...'])
    return '\n'.join(lines[:body_start] + [f'{indent}...'])
```

<!-- #_0ccc3c61 code -->
```python
def _class_chunk(atok, node, level):
    "Return disclosed source for class `node` at `level`"
    if level == 3: return atok.get_text(node)
    start_line = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_start = node.body[0].lineno - start_line
    lines = atok.get_text(node).splitlines()
    hdr = '\n'.join(lines[:body_start])
    body = []
    for i,n in enumerate(node.body):
        if isinstance(n, functypes): body.append(stub_fn(atok, n, level))
        elif isinstance(n, ast.Expr):
            if i == 0 and _is_docstring(n) and level >= 2: body.append(' '*n.col_offset + atok.get_text(n))
        else: body.append(' '*n.col_offset + atok.get_text(n))
    return hdr + '\n' + '\n'.join(body)

def _compound_chunk(atok, node, level):
    "Return disclosed source for compound statement `node` at `level`"
    if level == 3: return atok.get_text(node)
    text = atok.get_text(node)
    fns = sorted([n for n in ast.walk(node) if n is not node and isinstance(n, (*functypes, ast.ClassDef))],
                 key=lambda n: (n.lineno, n.col_offset), reverse=True)
    for n in fns:
        stub = stub_fn(atok, n, level) if isinstance(n, functypes) else _class_chunk(atok, n, level)
        lines = stub.splitlines()
        if lines[0].startswith(' ' * n.col_offset): lines[0] = lines[0][n.col_offset:]
        text = text.replace(atok.get_text(n), '\n'.join(lines), 1)
    return text
```

<!-- #_d27baedb code -->
```python
def disclose(src:str, level=1, strip_private=False) -> str:
    "Disclose Python source at `level` (1=sigs, 2=sigs+docs, 3=full)"
    if not src.strip(): return ''
    try:
        atok = asttokens.ASTTokens(src, parse=True)
        if strip_private: _strip_private(atok.tree)
        chunks = []
        for i,node in enumerate(atok.tree.body):
            if isinstance(node, functypes): chunks.append(stub_fn(atok, node, level))
            elif isinstance(node, ast.ClassDef): chunks.append(_class_chunk(atok, node, level))
            elif isinstance(node, _compound_types): chunks.append(_compound_chunk(atok, node, level))
            elif isinstance(node, ast.Expr) and not (i == 0 and _is_docstring(node) and level >= 2): pass
            elif isinstance(node, (ast.Import, ast.ImportFrom)) and level == 1: pass
            else: chunks.append(atok.get_text(node))
        return _postprocess('\n\n'.join(chunks))
    except SyntaxError as e:
        warnings.warn(f'Syntax error in source, returning raw: {e}')
        return src
```

<!-- #_1335cdb7 note -->
### dialog and notebook disclosure
Integrates per-cell `disclose` with solveit dialog and notebook file workflows. Three inclusion tiers: `'exported'` (module API), `'echo'` (API + doc examples), `'all'`.

<!-- #_20fca012 code -->
```python
_include_filter = dict(
    exported=lambda m: is_exported(m),
    echo=lambda m: is_exported(m) or is_echo(m),
    all=lambda m: True)

def _compact(s): return re.sub(r'\n{2,}', '\n', s).strip()

def _xml_cell(content, **attrs):
    attr_str = ''.join(f' {k}="{v}"' for k,v in attrs.items())
    return f'<cell{attr_str}>{content}</cell>'
```

<!-- #_62e97602 code -->
```python
async def dialog_code(dname='', include='all'):
    "Return list of (id, source) tuples for code cells filtered by `include`"
    msgs = await find_msgs_x(dname=dname, msg_type='code', include_output=False)
    return [(m.id, m.content) for m in msgs if _include_filter[include](m)]
```

<!-- #_ff8e4558 code -->
```python
def _disclose_srcs(srcs, level=1, strip_private=False, fmt='text', dname=''):
    "Disclose list of (id, source) tuples at `level`"
    chunks = [(mid, disclose(src, level, strip_private)) for mid,src in srcs]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(escape(src), id=mid) for mid,src in chunks)
        return f'<dialog level="{level}" path="{dname or "."}">\n{cells}\n</dialog>'
    if fmt == 'json':
        cells = [dict(id=mid, source=src) for mid,src in chunks]
        return json.dumps(dict(level=level, path=dname or '.', cells=cells), indent=2)
    return _compact('\n\n'.join(src for _,src in chunks))

async def disclose_dialog(dname='', level=1, include='all', strip_private=False, fmt='text'):
    "Disclose dialog `dname` at `level` filtering by `include`"
    srcs = await dialog_code(dname=dname, include=include)
    return _disclose_srcs(srcs, level=level, strip_private=strip_private, fmt=fmt, dname=dname)

def disclose_nb(path, level=1, strip_private=False, fmt='text'):
    "Return disclosed source of all code cells in notebook at `path`"
    nb = json.loads(Path(path).read_text())
    srcs = [(c.get('id',''), c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for c in nb['cells'] if c['cell_type'] == 'code']
    return _disclose_srcs(srcs, level=level, strip_private=strip_private, fmt=fmt, dname=str(path))
```

<!-- #_f48c816e note -->
### tests

<!-- #_882fc502 note -->
### Code disclosure: rationale, decisions, current state
Progressive disclosure of Python source code at three levels modelled on `.pyi` stub files. Per-cell, preserving message boundaries and dialog narrative structure.

<!-- #_e7e915df note -->
## distill
Utilities for reading and writing message content across sessions.

<!-- #_239b8e2a note -->
### filter messages v0
Filters a list of messages down to those relevant for export or documentation: exported (non-hidden) code cells, note headers, and cells with an echo directive.

<!-- #_32c810d5 code -->
```python
async def filter_msgs_v0(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Return notes, exported messages not hidden and code cells that have echo directive."
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return [m for m in msgs if (is_exported(m) and not is_hidden(m)) or is_header(m) or is_echo(m)]
```

<!-- #_69a9adb3 note -->
### digest dialog
Format a filtered digest of dialog messages for AI consumption. Supports `xml` (preferred), `md`, and `json` formats. Headers are progressively disclosed via `clamp_note` to keep digests compact. YAML frontmatter is prepended via `mk_frontmatter`.

<!-- #_643a905d code -->
```python
_pmark = '**🤖Reply🤖**<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n'
```

<!-- #_d2685927 code -->
```python
def _mk_tag(m, mtyp, ids):
    # parts = ([f'#{m.id}'] if ids else []) + [mtyp] + (['echo'] if echo else []) + (['skipped'] if m.skipped else []) + (['pinned'] if m.pinned else [])
    parts = ([f'#{m.id}'] if ids else []) + [mtyp]
    return ' '.join(parts)
```

<!-- #_d2bc045a code -->
```python
def output2md(c, langid='output'):
    return f'''```{langid}\n{c}\n```''' if c else ''
```

<!-- #_ff4409be code -->
```python
def strip_directives(c): return ('\n'.join(l for l in c.splitlines() if not l.startswith('#|'))).strip()
```

<!-- #_9ba03aa3 code -->
```python
_striptag_pats = (
    re.compile(r'^#\|\W*\w+: \w+\W*$\n?', re.MULTILINE),
    re.compile(r'^<!-- \w+: \w+ -->\s*\n?', re.MULTILINE))

def strip_tags(c):
    for pat in _striptag_pats: c = pat.sub('', c)
    return c.strip()
```

<!-- #_24b7596b code -->
```python
def _xml_msg(m:MsgX, ids, output, hdr_clamp):
    mtyp = m.msg_type
    c, attrs = '', (f' id="{m.id}"' if ids else '') + f' type="{mtyp}"'
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = escape(strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content))
            if c: c = f'<content>{c}</content>'
        case 'code':
            src = escape(strip_directives(m.content))
            out = f'\n  <output>{escape(m.output)}</output>' if output and m.output else ''
            if src or out: c = f'\n  <source>{src}</source>{out}\n'
        case 'prompt':
            p = escape(strip_tags(strip_directives(m.content)))
            r = f'\n  <response>{escape(m.output)}</response>' if m.output else ''
            if p or r: c = f'<prompt>{p}</prompt>{r}'
        case 'raw':
            c = escape(strip_tags(strip_directives(m.content)))
            if c: c = f'<msg{attrs}><content>{c}</content></msg>'
    return f'<msg{attrs}>{c}</msg>' if c else ''
```

<!-- #_d183b554 code -->
```python
def _md_msg(m, ids, output, hdr_clamp):
    mtyp, c = m.msg_type, None
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            c = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            c = f'```python\n{strip_directives(m.content)}\n```'
            if output and m.output: c += f'\n{output2md(m.output)}'
        case 'prompt':
            c = strip_tags(strip_directives(m.content))
            if output and m.output: c += f'\n{output2md(m.output, 'response')}'
        case 'raw': c = strip_tags(strip_directives(m.content))
    return f'<!-- {_mk_tag(m, mtyp, ids)} -->\n{c}\n' if c else ''
```

<!-- #_39d5c5d5 code -->
```python
def _json_msg(m, expd, echo, ids, output, hdr_clamp):
    mtyp = m.msg_type
    d = dict(type=mtyp)
    if ids: d['id'] = m.id
    match mtyp:
        case 'note':
            clamp = not m.exported_x and m.header_x
            d['content'] = strip_tags(clamp_note(m.content, hdr_clamp) if clamp else m.content)
        case 'code':
            d['source'] = strip_directives(m.content)
            if output and m.output: d['output'] = m.output
        case 'prompt':
            d['prompt'] = strip_tags(strip_directives(m.content))
            d['response'] = m.output
        case 'raw': d['content'] = strip_tags(strip_directives(m.content))
    return d
```

<!-- #_1fe5afdb code -->
```python
_convs = dict(
    xml="msg elements have id and type attributes; source/output are child elements of code msgs; prompt/response are child elements of prompt msgs",
    json="msgs is a list of objects with type and id fields; code objects have source and output fields; prompt objects have prompt and response fields; others have content",
    md="<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response")
```

<!-- #_c02cd206 code -->
```python
def mk_frontmatter(path='', description='', n_msgs=0, n_exported=0, fmt='xml'):
    from datetime import datetime, timezone
    return ("---\n"
"digest: solveit-dialog\n"
f"path: {path}\n"
f"exported_at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
f"description: {description!r}\n"
f"stats: {{total: {n_msgs}, exported: {n_exported}}}\n"
f"conventions: '{_convs[fmt]}'\n"
"---\n"
)
```

<!-- #_37710188 code -->
```python
def exportmsgs(
    msgs:list[Msg],
    hdr_clamp='header+',  # header level
    output=True,  # include code messages output
    ids=True,  # include message ids in cell comments
    frontmatter:dict=None,  # if present, call `mk_frontmatter` with this dict as args
    fmt='xml',  # 'xml'|'md'|'json'
):
    "Export messages as a digest for AI consumption"
    fn = {'xml': _xml_msg, 'md': _md_msg, 'json': _json_msg}[fmt]
    parts = [p for m in msgs if (p := fn(m, ids, output, hdr_clamp))]
    if fmt == 'json': res = json.dumps({'msgs': parts}, indent=2)
    else:
        body = '\n'.join(parts)
        res = f'<digest>\n{body}\n</digest>' if fmt=='xml' else body
    if frontmatter:
        frontmatter.update(n_exported=len(parts))
        fm = mk_frontmatter(**frontmatter)
    return f"{fm}\n{res}" if frontmatter else res
```

<!-- #_1b1594d4 note -->
Format a filtered, clamped digest of dialog messages for AI consumption in other sessions. Three output formats are supported: `xml` (preferred — unambiguous structure), `md`, and `json`, each implemented by a parallel `_xml_msg`/`_md_msg`/`_json_msg` function dispatched by `exportmsgs`.

**Clamping:** note content is progressively disclosed via `clamp_note(level)` — from header-only to header+paragraph — unless the note is fully exported. This keeps digests compact while preserving section structure and intent.

**Frontmatter:** `mk_frontmatter` prepends YAML metadata (path, timestamp, stats, format conventions) so the consuming LLM knows what it's looking at.

<!-- #_3fc335de note -->
### filter messages v1

<!-- #_dde7bb2c note -->
`find_msgs` is a staple of SolveIt, a very powerful introspection tool. But it have some limitations (see above how it doesn't honor explicit `nbdev` export directives) and I need more search granularity (e.g., to improve `exportmsgs`).

We're going to implement `filter_msgs_v1`, a wrapper over `find_msgs_x` with progressively improved search semantics. With no arguments, `filter_msgs_v1` return an empty list.

Let's start with `nbdev` export directives: if given `exported=True`, `filter_msgs_v1` should return **all** exported messages, those marked with SolveIt UI toggle **and** the ones with `nbdev` exported directives.

<!-- #_7800f21f code -->
```python
_meta_fns = dict(exported=is_exported, echo=is_echo, header=is_header, hidden=is_hidden, skipped=is_skipped)
```

<!-- #_187cb632 code -->
```python
async def get_section(heading:str, msgs:list[MsgX]=None, dname='') -> list[MsgX]:
    "Return messages in section whose header matches heading"
    if not msgs: return await find_msgs_x(header_section=heading, include_skipped=True, dname=dname)
    lvl = len(re.match(r'^(#+)', heading)[1])
    for i,m in enumerate(msgs):
        if m.msg_type != 'note' or not m.content.startswith(heading): continue
        res = [m]
        for m2 in msgs[i+1:]:
            hs = re.match(r'^(#+)', m2.content) if m2.msg_type == 'note' else None
            if hs and len(hs[1]) <= lvl: break
            res.append(m2)
        return res
    return []
```

<!-- #_c1f52e60 code -->
```python
def _parse_terms(v):
    terms = ([v] if isinstance(v, str) else v) or []
    return [t for t in terms if not t.startswith('!')], [t[1:] for t in terms if t.startswith('!')]
```

<!-- #_df65ab9d note -->
### content search
Regex filter on message content. Simple, composable, immediately useful for "find all cells referencing symbol foo".

<!-- #_67824193 code -->
```python
def _content_match(m, pat, use_case, use_regex):
    flags = (re.DOTALL|re.MULTILINE)|(0 if use_case else re.IGNORECASE)
    return bool(re.search(pat, m.content, flags) if use_regex else (pat in m.content if use_case else pat.lower() in m.content.lower()))
```

<!-- #_963c14a0 note -->
### section map
Builds a lookup from each markdown section header to the messages that belong to that section, including nested subsections. It lets us treat a dialog like a structured document and ask, “which messages are in this section?”

<!-- #_00ff0767 code -->
```python
def _header_level(m):
    if m.msg_type != 'note' or not m.content.startswith('#'): return 0
    return len(re.match(r'^(#+)', m.content)[1])
```

<!-- #_56979fd2 code -->
```python
def section_map(msgs):
    "Return dict mapping full header line -> list of msg ids in that section (inclusive of header)"
    res, cur_hdrs = {}, []
    for m in msgs:
        lvl = _header_level(m)
        if lvl:
            cur_hdrs = [(l,h) for l,h in cur_hdrs if l < lvl]
            hdr = re.match(r'^(#+[^\n]*)', m.content)[1]
            cur_hdrs.append((lvl, hdr))
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
        else:
            for _,h in cur_hdrs: res.setdefault(h, []).append(m.id)
    return res
```

<!-- #_9c991d36 note -->
### filter messages v1 wrap up
`filter_msgs_v1` is the dialog query helper for selecting messages by ids, sections, message type, metadata, and content, with support for negation and dialog-order results.

<!-- #_16a9f89d code -->
```python
def _resolve_sections(msgs, patterns):
    "Return set of msg ids matching any of the section patterns (supports regex via section_map)"
    sm = section_map(msgs)
    ids = set()
    for p in patterns:
        for k, v in sm.items():
            if re.fullmatch(p, k, re.IGNORECASE): ids |= set(v)
    return ids
```

<!-- #_073fd07f code -->
```python
async def filter_msgs_v1(
    msgs:list[Msg]=None,
    ids:str|list=None,
    section:str|list=None,
    msg_type:str|list=None,
    metadata:str|list=None,
    re_pattern:str='',
    use_case:bool=False,
    use_regex:bool=True,
    dname:str='',
) -> list[MsgX]:
    "Filter messages with union semantics and ! negation; result in dialog order."
    if not any([ids, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    ids_l = ([ids] if isinstance(ids, str) else ids) or []
    sec_pos, sec_neg = _parse_terms(section)
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([ids_l, sec_pos, mt_pos, md_pos, (pat_pos and not pat_neg)])
    pos = set(ids_l)
    pos |= _resolve_sections(msgs, sec_pos)
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    neg |= _resolve_sections(msgs, sec_neg)
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]
```

<!-- #_bb1f6ca6 note -->
### filter messages v2
Query helper for selecting dialog messages by id or section, then filtering by message type, metadata, or content pattern. Supports negation and preserves dialog order.

<!-- #_9899a2d5 code -->
```python
def _scope_msgs(msgs, ids, section):
    "Narrow msgs to those matching ids and/or section patterns."
    if ids is None and not section: return msgs
    scoped = set()
    if ids is not None: scoped |= {ids} if isinstance(ids, str) else set(ids)
    if section:
        secs = [section] if isinstance(section, str) else section
        scoped |= _resolve_sections(msgs, secs)
    return [m for m in msgs if m.id in scoped]
```

<!-- #_db9f095e code -->
```python
def _filter_pos_neg(msgs, msg_type=None, metadata=None, re_pattern='', use_case=False, use_regex=True):
    "Apply union+negation filter to msgs. If no positive terms, return msgs as-is."
    mt_pos, mt_neg = _parse_terms(msg_type)
    md_pos, md_neg = _parse_terms(metadata)
    pat_pos, pat_neg = (re_pattern[1:], True) if re_pattern.startswith('!') else (re_pattern, False)
    has_pos = any([mt_pos, md_pos, pat_pos and not pat_neg])
    if not has_pos and not any([mt_neg, md_neg, pat_neg]): return msgs
    pos = set()
    if mt_pos: pos |= {m.id for m in msgs if m.msg_type in mt_pos}
    if md_pos: pos |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_pos)}
    if pat_pos and not pat_neg: pos |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    neg = set()
    if mt_neg: neg |= {m.id for m in msgs if m.msg_type in mt_neg}
    if md_neg: neg |= {m.id for m in msgs if any(_meta_fns[k](m) for k in md_neg)}
    if pat_neg: neg |= {m.id for m in msgs if _content_match(m, pat_pos, use_case, use_regex)}
    base = pos if has_pos else {m.id for m in msgs}
    return [m for m in msgs if m.id in base - neg]
```

<!-- #_ad25c0ee code -->
```python
async def filter_msgs_v2(
    msgs:list[Msg]=None, # List of messages to filter or None to use all messages.
    ids:str|list=None, # Message ids to include (union with section).
    section:str|list=None, # Section patterns to include (union with ids).
    msg_type:str|list=None, # Message types to include.
    metadata:str|list=None, # Metadata keys to to filter on ('exported', 'echo', 'header', 'hidden').
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used).
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
    dname:str='', # Dialog to get messages for; defaults to current dialog.
) -> list[MsgX]:
    "Two-phase filter: scope by ids/section, then OR-filter by type/metadata/pattern with ! negation."
    if not any([ids is not None, section, msg_type, metadata, re_pattern]): return []
    msgs = msgs or await find_msgs_x(include_skipped=True, dname=dname)
    return _filter_pos_neg(_scope_msgs(msgs, ids, section), msg_type, metadata, re_pattern, use_case, use_regex)
```

<!-- #_3e639406 code -->
```python
syn = [mk('a', '## visible'), mk('b', '## skipped', skipped=1), mk('c', 'x=1', 'code')]
ms = await filter_msgs_v2(syn, metadata=['header', '!skipped'])
test_eq(ms, [syn[0]])
```

<!-- linkedto: _9cf954a5 -->

Before going on, I know is difficult, you can't forget what's in your context, but try. Take a look a `md` result above. Is a good starting point for the future discussion? Does approach the goal of generating "good" context from a dialog? Being "good context" what we give to an AI to understand what a dialog is about. It could be to create a handoff for forking a dialog or compacting parts of a dialog, or using some functions from a dialog in other dialog.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — it’s a **good starting point**, and it’s already much better than just dumping the whole dialog. It has the main ingredients of “good AI context”:

- **Frontmatter**: path, timestamp, stats, format conventions.
- **Stable message ids**: useful for referring back to exact cells.
- **Section structure**: the AI can see the shape of the dialog.
- **Selected code**: the important helpers are present.
- **Narrative notes**: enough prose is included to understand intent, not just implementation.
- **Dialog order preserved**: very important; the code evolution remains understandable.

But I’d call this a **good raw digest**, not yet a great handoff. For a handoff/fork/compaction context, I think we need one more layer on top: a short human-written or generated “orientation” section before the digest.

Something like:

```markdown
## Orientation

This dialog develops utilities for querying and exporting Solveit dialog messages.

The current preferred query helper is `filter_msgs_v2`, which uses two phases:

1. Scope messages by `ids` and/or `section`
2. Filter the scoped messages by `msg_type`, `metadata`, and/or `re_pattern`

Positive filter terms are ORed. `!`-prefixed terms are subtracted globally within the scoped set.

`filter_msgs_v1` is retained for study but is not the preferred interface, because its union semantics across sections and filters were harder to reason about.

Important supporting helpers:

- `section_map(msgs)` builds section membership from note headers
- `_resolve_sections(msgs, patterns)` resolves section regex patterns to message ids
- `_scope_msgs(msgs, ids, section)` narrows the message list before filtering
- `_filter_pos_neg(...)` applies OR-positive / subtract-negative filtering
- `exportmsgs(...)` serializes selected messages to md/xml/json digests
- `clamp_note(...)` progressively discloses markdown notes

Known design decision: Solveit sections are note messages whose first line starts with `#`.
```

Then include the generated digest below it.

The current `md` result is strong, but I see a few issues:

1. **It is still too broad for some future uses**

   For example, it includes the whole “Code message disclosure” machinery. That is relevant to context generation overall, but maybe not to someone only continuing `filter_msgs_v2`.

   For a fork, broad is okay.  
   For “use these functions in another dialog”, we probably want a smaller API/dependency bundle.

2. **It includes exploratory versions that could confuse the next AI**

   `filter_msgs_v1` appears with full code, then `filter_msgs_v2` appears later. That’s useful historically, but the handoff should explicitly say:

   > `filter_msgs_v2` is preferred; `filter_msgs_v1` is retained as exploratory/reference material.

3. **Important tests-as-documentation are not fully represented**

   The synthetic tests you wrote are excellent context. They explain the semantics better than prose. For this kind of helper, I’d want the handoff to include a compact “semantic examples” block:

   ```python
   ms = await filter_msgs_v2(msgs, section='## distill', metadata=['exported', 'echo', 'header', '!hidden'])
   ```

   and maybe minimal synthetic examples for:

   - negation only
   - scoped section + metadata
   - `msg_type='!code'`
   - `re_pattern='!foo'`

4. **For “use functions from a dialog in another dialog”, this is not enough**

   The digest explains the functions, but it is not necessarily an executable dependency bundle. For that use case, the context should also include:

   - required imports
   - exported functions/classes only
   - dependency order
   - maybe `disclose(..., level=2)` rather than full source
   - examples/tests showing expected behavior

So I’d frame the current result as:

> A good structured digest for AI reading, suitable as raw material for handoff or compaction, but it should be preceded by a short orientation summary and optionally specialized depending on the target use case.

A useful next design distinction might be three export modes:

```python
handoff   -> summary + selected narrative + preferred API + examples
compact   -> preserve section narrative, reduce implementation detail
reuse     -> imports + dependency-ordered code + minimal tests
```

The `md` digest above is closest to **compact**, with some **handoff** qualities. To become a really good handoff, it needs the explicit orientation layer and clearer “current preferred state” markers.

<details class='token-usage-details' markdown='1'><summary>$0.0091</summary>

`total=54,443 | in=52,889 | out=1,554 | cached=4.6% | reasoning=516 | searches=0 | $0.0091`

</details>


Of course, it was a test. I've got to sweep the whole dialog cleaning and marking messages. That isn't the final result at all. We're going to continue with this dialog.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly — the handoff quality is only as good as the curation upstream. The export is the easy part; the real work is the sweep: marking exports, writing good section notes, and deciding what the next session needs to know vs. what it can infer from code alone.

When you're ready to do that sweep, `filter_msgs_v2` with `section` + `metadata` is now a good tool for it — you can pull any section, inspect what's exported, and patch gaps.

<details class='token-usage-details'><summary>$0.0343</summary>

`total=64,471 | in=64,360 | out=111 | cached=99.1% | cache_new=378 | searches=0 | $0.0343`

</details>


The two-phase design works well — 33 messages, cleanly scoped and filtered. Now let's draft the handoff markdown:

```python
handoff_msgs = await filter_msgs_v2(msgs[:this_idx], 
    section=['## message utils', '## dialog utils'],
    metadata=['exported', 'echo', 'header', '!hidden'])

fmt = 'md'
handoff = exportmsgs(handoff_msgs, hdr_clamp='header++', fmt=fmt, frontmatter=dict(
    path=find_dname(), 
    description='Handoff: filter_msgs_v1/v2 and section filtering in pote.dutil',
    n_msgs=len(msgs), fmt=fmt))

Path('data/handoff_filter_msgs.md').write_text(handoff)
print(f"{len(handoff)} chars, {len(handoff_msgs)} msgs")
```

Great! Ok, what next? Be wild!

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few directions, roughly by ambition:

1. **Content search** — `contains='pattern'` regex filter on message content. Simple, composable, immediately useful for "find all cells referencing symbol `foo`".

2. **Token budget** — `max_tokens=N`: take messages in dialog order until budget exhausted. Critical for fitting digests into LLM context windows without manual slicing.

3. **Wire into `exportmsgs`** — accept `filter_msgs_v1` kwargs directly so callers don't need to pre-filter; deprecate `filter_msgs_v0`.

4. **Dependency tracking** — given a set of code cells, walk backwards through the dialog to find all cells that define symbols they reference. True isolation: a slice that can run correctly in a fresh kernel. This is the wildest one — essentially a lightweight static analyser over dialog cells.

The last one is the most powerful and would unlock real "extract section to new dialog" workflows. But #1 or #2 are good warm-ups.

<details class='token-usage-details'><summary>$0.0512</summary>

`total=44,318 | in=43,762 | out=556 | cached=91.5% | cache_new=3,458 | searches=0 | $0.0512`

</details>


## token budget

Let's think about the content budget. That would be very good to have but I don't see any way of making it deterministic. I don't want to lose whole messages or truncate content, so we would have to rely on LLM compaction, always tricky and dangerous. Ideas, suggestions?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The constraint "no lost messages, no truncated content" is very tight. A few angles that are deterministic:

1. **Progressive note disclosure** — already in `clamp_note`. Dialing down from `header++` to `header+` to `header` is lossless (full content still in dialog) and can shave tokens significantly for long notes.

2. **Metadata-only mode** — return just ids + types + first line of content as a table of contents, letting the consumer request full sections on demand. But this requires a back-and-forth protocol.

Which of these feels closest to what you need?

<details class='token-usage-details'><summary>$0.0363</summary>

`total=50,548 | in=50,000 | out=548 | cached=99.4% | cache_new=283 | searches=0 | $0.0363`

</details>


Well, we have the progressive note disclosure strategy working with headers, `clamp_note`, used by `exportmsgs`, and we got also code disclosure strategy, `disclose`, `_disclose_srcs`, `disclose_dialog`, `disclose_nb`. Now is a good time to merge both strategys in order to get a dialed-in mechanism to create digests of a dialog at different levels of disclosure for different needs: handoff, discovery, compaction, etc. Let's discuss.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — this feels like the right convergence point. I’d separate the system into three independent layers:

1. **Selection**: which messages are in the digest  
   `filter_msgs_v2`, `section_map`, ids, sections, metadata, content search.

2. **Disclosure**: how much of each selected message is shown  
   `clamp_note` for notes, `disclose` for code, maybe response/output disclosure later.

3. **Serialization**: how the disclosed messages are formatted  
   `exportmsgs` to `md` / `xml` / `json`.

Right now `exportmsgs` mostly does layers 2 and 3 together, but only partially: it clamps notes, while code is exported as full source after directive stripping. The natural next design is to make disclosure explicit and uniform.

A useful mental model:

```python
profile = dict(note='header++', code=2, output=False, prompt='full', response='full')
```

Then `exportmsgs` can receive either explicit levels or a named profile.

I’d avoid thinking about this as “compression” at first. It’s more like **progressive rendering**:

| Msg type | Low disclosure | Medium disclosure | High disclosure |
|---|---|---|---|
| note | header | header+ / header++ | full |
| code | signature skeleton | sigs + docs/comments/imports | full source |
| output | absent | short / selected | full |
| prompt | prompt only | prompt + short response | prompt + full response |
| raw | absent / first line | full | full |

For our immediate needs, I’d start with just notes and code, because those are already implemented and deterministic.

The key improvement is: **budgeting can be deterministic if we use a disclosure ladder instead of truncation.**

For example:

```python
_profiles = dict(
    discovery=dict(note='header', code=1, output=False),
    map=dict(note='header+', code=1, output=False),
    compact=dict(note='header++', code=1, output=False),
    handoff=dict(note='header++', code=2, output=False),
    reuse=dict(note='full', code=3, output=False),
    archive=dict(note='full', code=3, output=True))
```

Then a budgeted export can do:

1. Generate digest with richest desired profile
2. Count tokens
3. If too large, step down the profile ladder
4. If even the lowest profile is too large, split deterministically by top-level sections

No LLM compaction needed. No arbitrary truncation. No silent message loss.

The “split by sections” fallback is important. If the user says “don’t lose whole messages or truncate content”, then the lowest safe fallback is not “cut text”, but:

```text
digest part 1/4: sections A-B
digest part 2/4: sections C-D
digest part 3/4: sections E-F
digest part 4/4: section index / manifest
```

Each part remains internally complete at its chosen disclosure level.

I’d also distinguish two kinds of “complete”:

- **Complete message coverage**: every selected message appears somehow, maybe just as id/type/header/skeleton.
- **Complete content coverage**: every selected message appears with full original content.

For AI context, complete message coverage is usually enough for discovery/handoff. Complete content coverage is only needed for reuse/archive.

So the digest should probably include frontmatter like:

```yaml
disclosure:
  note: header++
  code: 2
  output: false
coverage:
  messages: complete
  content: partial
```

That makes the digest honest. The consuming AI knows whether it is seeing the whole source or a disclosed summary.

The design I’d aim for:

```python
def disclose_msg(m, note_level='header++', code_level=2, output=False):
    ...
```

Then `exportmsgs` becomes simpler:

```python
def exportmsgs(msgs, profile=None, note_level='header++', code_level=3, output=True, ...):
    msgs = [disclose_msg(m, note_level, code_level, output) for m in msgs]
    ...
```

Or maybe cleaner:

```python
def disclose_msgs(msgs, note_level='header++', code_level=2, output=False): ...
def exportmsgs(msgs, fmt='md', ...): ...
```

I slightly prefer the second approach. Disclosure transforms messages; export serializes messages. That keeps the layers clean.

The profile names should correspond to real workflows:

- **discovery**: “What is in this dialog?”
  - notes: `header`
  - code: `1`
  - output: `False`

- **handoff**: “Continue this work in a new session.”
  - notes: `header++`
  - code: `2`
  - output: usually `False`
  - include tests/examples if marked `echo`

- **compact**: “Preserve the reasoning and API shape.”
  - notes: `header++`
  - code: `1` or `2`
  - output: `False`

- **reuse**: “Use these functions elsewhere.”
  - notes: `header+`
  - code: `3` for selected/exported code
  - output: maybe tests only

- **archive**: “Lossless-ish record.”
  - notes: full
  - code: full
  - output: full

One subtlety: code level probably shouldn’t always be global. For handoff, we may want:

- exported API: code level 2 or 3
- tests/examples: full if `echo`
- private helpers: level 1 or omitted
- exploratory dead ends: note header only, or omitted by selection

But I wouldn’t implement that yet. First make one global note level and one global code level work.

The next small implementation step should probably be a pure helper that doesn’t touch `exportmsgs` yet:

```python
def disclose_msg(m:MsgX, note_level='header++', code_level=2, output=False) -> MsgX:
    ...
```

It returns a shallow copy of the message with disclosed `.content` and maybe `.output=''` when outputs are off. Then we can visually test on one synthetic note and one synthetic code message before wiring it into `exportmsgs`.

<details class='token-usage-details' markdown='1'><summary>$0.0096</summary>

`total=56,584 | in=54,737 | out=1,847 | cached=4.4% | reasoning=516 | searches=0 | $0.0096`

</details>


Let's start with a pure message-level disclosure helper. It returns a copied `MsgX`, so the original message list stays untouched.

In [ ]:
def disclose_msg(m:MsgX, note_level='header++', code_level=2, output=False) -> MsgX:
    "Return a disclosed copy of message `m`."
    res = msgx(AD(m.copy()))
    if res.msg_type == 'note' and note_level != 'full': res.content = clamp_note(res.content, note_level)
    if res.msg_type == 'code' and code_level != 'full': res.content = disclose(strip_directives(res.content), code_level)
    if not output: res.output = ''
    return res

And a tiny visual/test check:

In [ ]:
syn = [
    mk('n', '## Heading\n\nFirst paragraph.\n\nSecond paragraph.'),
    mk('c', 'def foo(x:int) -> int:\n    "Add one."\n    return x+1', 'code', output='ignored')
]
d = [disclose_msg(m, note_level='header++', code_level=2) for m in syn]
test_eq(d[0].content, '## Heading\nFirst paragraph.')
test_eq(d[1].content, 'def foo(x:int) -> int:\n    "Add one."\n    ...')
test_eq(d[1].output, '')
[d[0].content, d[1].content]

['## Heading\nFirst paragraph.',
 'def foo(x:int) -> int:\n    "Add one."\n    ...']

Next, a thin list-level wrapper so disclosure can be applied before serialization:

In [ ]:
#| export
def disclose_msgs(msgs:list[MsgX], note_level='header++', code_level=2, output=False) -> list[MsgX]:
    "Return disclosed copies of `msgs`."
    return [disclose_msg(m, note_level=note_level, code_level=code_level, output=output) for m in msgs]

And a small check that the originals are untouched:

In [ ]:
orig = syn[1].content
d = disclose_msgs(syn, note_level='header', code_level=1)
test_eq([m.content for m in d], ['## Heading', 'def foo(x: int) -> int: ...'])
test_eq(syn[1].content, orig)
[m.content for m in d]

['## Heading', 'def foo(x: int) -> int: ...']

Next, add named profiles as a thin layer over `disclose_msgs`:

In [ ]:
#| export
_profiles = dict(
    discovery=dict(note_level='header', code_level=1, output=False), 
    compact=dict(note_level='header++', code_level=1, output=False), 
    handoff=dict(note_level='header++', code_level=2, output=False), 
    reuse=dict(note_level='header+', code_level=3, output=False), 
    archive=dict(note_level='full', code_level='full', output=True)
)

In [ ]:
def disclose_profile(msgs:list[MsgX], profile='handoff', **kwargs) -> list[MsgX]:
    "Return disclosed copies of `msgs` using a named profile."
    kw = {**_profiles[profile], **kwargs}
    return disclose_msgs(msgs, **kw)

And a small check:

In [ ]:
d = disclose_profile(syn, 'discovery')
test_eq([m.content for m in d], ['## Heading', 'def foo(x: int) -> int: ...'])

d = disclose_profile(syn, 'archive')
test_eq(d[0].content, syn[0].content)
test_eq(d[1].content, syn[1].content)
test_eq(d[1].output, 'ignored')

[d[0].content, d[1].content, d[1].output]

['## Heading\n\nFirst paragraph.\n\nSecond paragraph.',
 'def foo(x:int) -> int:\n    "Add one."\n    return x+1',
 'ignored']

Next small step: avoid double-clamping disclosed notes by teaching the serializers that `hdr_clamp='full'` means “do not clamp here”.

In [ ]:
def _maybe_clamp_note(m, hdr_clamp):
    "Clamp note content unless `hdr_clamp` is full."
    return m.content if hdr_clamp == 'full' or m.exported_x or not m.header_x else clamp_note(m.content, hdr_clamp)

Then patch just the note branch in the serializers. Start with markdown, since that is what we’re testing visually:

In [ ]:
def _md_msg(m, ids, output, hdr_clamp):
    mtyp, c = m.msg_type, None
    match mtyp:
        case 'note': c = strip_tags(_maybe_clamp_note(m, hdr_clamp))
        case 'code':
            c = f'```python\n{strip_directives(m.content)}\n```'
            if output and m.output: c += f'\n{output2md(m.output)}'
        case 'prompt':
            c = strip_tags(strip_directives(m.content))
            if output and m.output: c += f'\n{output2md(m.output, "response")}'
        case 'raw': c = strip_tags(strip_directives(m.content))
    return f'<!-- {_mk_tag(m, mtyp, ids)} -->\n{c}\n' if c else ''

Quick check:

In [ ]:
d = disclose_profile(syn, 'handoff')
md = exportmsgs(d, hdr_clamp='full', output=False, fmt='md')
test_is('Second paragraph' in md, False)
test_is('First paragraph' in md, True)
print(md)

<!-- #n note -->
## Heading
First paragraph.

<!-- #c code -->
```python
def foo(x:int) -> int:
    "Add one."
    ...
```



Next I’d make `xml` and `json` honor the same `hdr_clamp='full'` contract, and fix the `json` serializer signature while we’re here.

In [ ]:
def _xml_msg(m:MsgX, ids, output, hdr_clamp):
    mtyp = m.msg_type
    c,attrs = '', (f' id="{m.id}"' if ids else '') + f' type="{mtyp}"'
    match mtyp:
        case 'note':
            c = escape(strip_tags(_maybe_clamp_note(m, hdr_clamp)))
            if c: c = f'<content>{c}</content>'
        case 'code':
            src = escape(strip_directives(m.content))
            out = f'\n  <output>{escape(m.output)}</output>' if output and m.output else ''
            if src or out: c = f'\n  <source>{src}</source>{out}\n'
        case 'prompt':
            p = escape(strip_tags(strip_directives(m.content)))
            r = f'\n  <response>{escape(m.output)}</response>' if output and m.output else ''
            if p or r: c = f'<prompt>{p}</prompt>{r}'
        case 'raw':
            c = escape(strip_tags(strip_directives(m.content)))
            if c: c = f'<content>{c}</content>'
    return f'<msg{attrs}>{c}</msg>' if c else ''

def _json_msg(m, ids, output, hdr_clamp):
    mtyp = m.msg_type
    d = dict(type=mtyp)
    if ids: d['id'] = m.id
    match mtyp:
        case 'note': d['content'] = strip_tags(_maybe_clamp_note(m, hdr_clamp))
        case 'code':
            d['source'] = strip_directives(m.content)
            if output and m.output: d['output'] = m.output
        case 'prompt':
            d['prompt'] = strip_tags(strip_directives(m.content))
            if output and m.output: d['response'] = m.output
        case 'raw': d['content'] = strip_tags(strip_directives(m.content))
    return d

In [ ]:
d = disclose_profile(syn, 'handoff')
xml = exportmsgs(d, hdr_clamp='full', output=False, fmt='xml')
js = json.loads(exportmsgs(d, hdr_clamp='full', output=False, fmt='json'))

test_is('Second paragraph' in xml, False)
test_is('First paragraph' in xml, True)
test_eq(js['msgs'][0]['content'], '## Heading\nFirst paragraph.')
test_eq(js['msgs'][1]['source'], 'def foo(x:int) -> int:\n    "Add one."\n    ...')

print(xml)

<digest>
<msg id="n" type="note"><content>## Heading
First paragraph.</content></msg>
<msg id="c" type="code">
  <source>def foo(x:int) -&gt; int:
    &quot;Add one.&quot;
    ...</source>
</msg>
</digest>


Next, I’d add a thin profile-aware export wrapper, without changing `exportmsgs` itself yet:

In [ ]:
#| export
def disclosure_kwargs(profile='handoff', **kwargs):
    "Return disclosure kwargs for `profile`."
    return {**_profiles[profile], **kwargs}

def disclose_profile(msgs:list[MsgX], profile='handoff', **kwargs) -> list[MsgX]:
    "Return disclosed copies of `msgs` using a named profile."
    return disclose_msgs(msgs, **disclosure_kwargs(profile, **kwargs))

In [ ]:
def exportmsgs_profile(msgs:list[MsgX], profile='handoff', fmt='md', ids=True, frontmatter=None, **kwargs):
    "Export messages after applying a disclosure profile."
    kw = disclosure_kwargs(profile, **kwargs)
    return exportmsgs(disclose_msgs(msgs, **kw), hdr_clamp='full', output=kw['output'], ids=ids, frontmatter=frontmatter, fmt=fmt)

And a tiny check:

In [ ]:
md = exportmsgs_profile(syn, 'handoff', fmt='md')
test_is('Second paragraph' in md, False)
test_is('return x+1' in md, False)
test_is('"Add one."' in md, True)

md = exportmsgs_profile(syn, 'archive', fmt='md')
test_is('Second paragraph' in md, True)
test_is('return x+1' in md, True)
test_is('ignored' in md, True)

print(md)

<!-- #n note -->
## Heading

First paragraph.

Second paragraph.

<!-- #c code -->
```python
def foo(x:int) -> int:
    "Add one."
    return x+1
```
```output
ignored
```



Next I’d make the disclosure level explicit in the digest frontmatter. Otherwise a future AI sees partial code/notes but has to infer whether they’re full or disclosed.

In [ ]:
def mk_frontmatter(path='', description='', n_msgs=0, n_exported=0, fmt='xml', profile='', disclosure=None):
    from datetime import datetime, timezone
    disc = f"disclosure: {json.dumps(disclosure)}\n" if disclosure else ''
    prof = f"profile: {profile!r}\n" if profile else ''
    return ("---\n"
"digest: solveit-dialog\n"
f"path: {path}\n"
f"exported_at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
f"description: {description!r}\n"
f"{prof}"
f"{disc}"
f"stats: {{total: {n_msgs}, exported: {n_exported}}}\n"
f"conventions: '{_convs[fmt]}'\n"
"---\n"
)

And patch the profile wrapper to pass that metadata through when frontmatter is provided:

In [ ]:
#| export
def exportmsgs_profile(msgs:list[MsgX], profile='handoff', fmt='md', ids=True, frontmatter=None, **kwargs):
    "Export messages after applying a disclosure profile."
    kw = disclosure_kwargs(profile, **kwargs)
    if frontmatter is not None: frontmatter = {**frontmatter, 'profile':profile, 'disclosure':kw}
    return exportmsgs(disclose_msgs(msgs, **kw), hdr_clamp='full', output=kw['output'], ids=ids, frontmatter=frontmatter, fmt=fmt)

Tiny check:

In [ ]:
md = exportmsgs_profile(syn, 'handoff', fmt='md', frontmatter=dict(path='test', description='profile test', n_msgs=len(syn), fmt='md'))
test_is('profile: ' in md, True)
test_is('disclosure: ' in md, True)
print(md.split('---')[1])


digest: solveit-dialog
path: test
exported_at: 2026-05-18T10:37:39+00:00
description: 'profile test'
profile: 'handoff'
disclosure: {"note_level": "header++", "code_level": 2, "output": false}
stats: {total: 2, exported: 2}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'



Next, let’s measure the actual budget impact of each profile on the same selected message set:

In [ ]:
profile_sizes = []
for k in _profiles:
    md = exportmsgs_profile(v2, k, fmt='md', frontmatter=dict(path=find_dname(), description=f'{k} profile budget check', n_msgs=len(v2), fmt='md'))
    profile_sizes.append(AD(profile=k, chars=len(md), tokens=count_tokens(md), msgs=len(v2)))
profile_sizes

[{'profile': 'discovery', 'chars': 8504, 'tokens': 2671, 'msgs': 84},
 {'profile': 'compact', 'chars': 13078, 'tokens': 3667, 'msgs': 84},
 {'profile': 'handoff', 'chars': 16510, 'tokens': 4466, 'msgs': 84},
 {'profile': 'reuse', 'chars': 31315, 'tokens': 8709, 'msgs': 84},
 {'profile': 'archive', 'chars': 35532, 'tokens': 9797, 'msgs': 84}]

Next deterministic piece: choose the richest profile that fits a token budget, and explicitly signal when even the smallest profile is too large.

In [ ]:
_profile_order = ['archive', 'reuse', 'handoff', 'compact', 'discovery']

def digest_for_profile(msgs, profile, fmt='md', frontmatter=None, **kwargs):
    fm = None if frontmatter is None else frontmatter.copy()
    return exportmsgs_profile(msgs, profile, fmt=fmt, frontmatter=fm, **kwargs)

def fit_digest(msgs, max_tokens, profiles=None, fmt='md', frontmatter=None, **kwargs):
    profiles = profiles or _profile_order
    rows = []
    for p in profiles:
        txt = digest_for_profile(msgs, p, fmt=fmt, frontmatter=frontmatter, **kwargs)
        rows.append(AD(profile=p, tokens=count_tokens(txt), chars=len(txt), text=txt))
    fits = [r for r in rows if r.tokens <= max_tokens]
    return fits[0] if fits else AD(profile=None, tokens=rows[-1].tokens, chars=rows[-1].chars, text=rows[-1].text, split_needed=True, tried=rows)

Tiny check against the profile sizes you just measured:

In [ ]:
fm = dict(path=find_dname(), description='budget fit check', n_msgs=len(v2), fmt='md')

r = fit_digest(v2, 5000, frontmatter=fm)
test_eq(r.profile, 'handoff')

r = fit_digest(v2, 3000, frontmatter=fm)
test_eq(r.profile, 'discovery')

r = fit_digest(v2, 2000, frontmatter=fm)
test_eq(r.profile, None)
test_is(r.split_needed, True)

r.profile, r.tokens, r.chars

(None, 2669, 8490)

In [ ]:
solveit_docs?

````python
def solveit_docs(
    
):

````



````
Full reference documentation for Solveit - use this to answer questions about how to use Solveit.
**NB**: The whole docs fit in LLM context, so read the whole thing, don't search/filter it. *Always* re-run rather than relying on truncated history or assumptions.
````



**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

**Type:** function

In [ ]:
dname_doc

"If `dname` is None, the current dialog is used (if any). If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**"

In [ ]:
dialoghelper_explain_dialog_editing?

````python
def dialoghelper_explain_dialog_editing(
    
)->str: # Detailed documention on dialoghelper dialog editing

````



````
Call this to get a detailed explanation of how dialog editing is done in dialoghelper.
**ALWAYS** call this first, if dialog editing has not previously occured in this session
````



**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

**Type:** function

Yes, that’s a very good idea. The frontmatter should stay compact, but it can point to a richer “explain this format” helper. That gives us:

- short digest metadata inline
- longer conceptual docs available on demand
- a stable function/tool name the AI can call when confused
- less pressure to over-explain every exported digest

I’d add a small exported helper first:

In [ ]:
def dialog_digest_docs() -> str:
    "Explain solveit dialog digest frontmatter, disclosure profiles, and usage."
    return """
# Solveit dialog digest docs

A dialog digest is a selected, disclosed, serialized view of a Solveit dialog for AI consumption.

## Frontmatter fields

- `digest`: digest kind, currently `solveit-dialog`
- `path`: source dialog path
- `exported_at`: UTC timestamp
- `description`: human-readable purpose of this digest
- `profile`: named disclosure profile, if used
- `disclosure`: exact note/code/output disclosure settings
- `stats.total`: number of messages in the source selection
- `stats.exported`: number of serialized messages in the digest
- `conventions`: serialization conventions for the chosen format
- `docs`: helper function to call for this explanation

## Disclosure profiles

- `discovery`: section/map overview; note headers and code skeletons
- `compact`: compact narrative; note header plus first paragraph and code skeletons
- `handoff`: good continuation context; notes plus code signatures/docstrings
- `reuse`: fuller source for using functions elsewhere
- `archive`: full notes, full code, and outputs

## Important semantics

Disclosure is deterministic. Messages are not randomly summarized, truncated, or rewritten by an LLM.

Partial disclosure means the digest may show only a header, skeleton, or signature-level view of a message. The original dialog remains the source of truth.

`profile` names are convenience presets. The exact `disclosure` field is authoritative.

## Selection vs disclosure vs serialization

Selection chooses which messages are included.

Disclosure chooses how much of each selected message is shown.

Serialization chooses the output format: markdown, XML, or JSON.

## Execution correctness requirement

disclosed code at every level should be valid Python that can be executed in the appropriate dialog/export context. Disclosure is not responsible for dependency isolation: callers must include the necessary import cells, private helpers, decorators, base classes, and other dependencies in the selected/exported message set. Full or partial isolation is handled separately by isolation tooling.

## Recommended AI behavior

Read the frontmatter first.

If `profile` is not `archive`, assume some content is intentionally disclosed rather than complete.

Use message ids to refer to exact source messages.

If more detail is needed, ask for a richer profile or a narrower section at a richer profile.
""".strip()

Then add a tiny `docs` line to frontmatter:

In [ ]:
def mk_frontmatter(path='', description='', n_msgs=0, n_exported=0, fmt='xml', profile='', disclosure=None, docs='dialog_digest_docs()'):
    from datetime import datetime, timezone
    disc = f"disclosure: {json.dumps(disclosure)}\n" if disclosure else ''
    prof = f"profile: {profile!r}\n" if profile else ''
    docs = f"docs: {docs!r}\n" if docs else ''
    return ("---\n"
"digest: solveit-dialog\n"
f"path: {path}\n"
f"exported_at: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
f"description: {description!r}\n"
f"{prof}"
f"{disc}"
f"{docs}"
f"stats: {{total: {n_msgs}, exported: {n_exported}}}\n"
f"conventions: '{_convs[fmt]}'\n"
"---\n"
)

This keeps the digest self-describing without bloating every export. The frontmatter says “there is a richer explanation available”, and the helper gives the AI enough conceptual grounding to interpret disclosed context correctly.

Next I’d lock down the new frontmatter/docs contract with a tiny test before moving on to section splitting or richer budget handling.

In [ ]:
md = exportmsgs_profile(syn, 'handoff', fmt='md', frontmatter=dict(path='test', description='docs test', n_msgs=len(syn), fmt='md'))
fm = md.split('---')[1]
test_is("docs: 'dialog_digest_docs()'" in fm, True)
test_is("profile: 'handoff'" in fm, True)
test_is('disclosure: ' in fm, True)
test_is('Disclosure profiles' in dialog_digest_docs(), True)
print(fm)


digest: solveit-dialog
path: test
exported_at: 2026-05-18T10:51:39+00:00
description: 'docs test'
profile: 'handoff'
disclosure: {"note_level": "header++", "code_level": 2, "output": false}
docs: 'dialog_digest_docs()'
stats: {total: 2, exported: 2}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'



## correctness note

Before continuing section splitting, let's consider one requirement that I don't know is explicit in the code disclosure sections: execution correctness. It was a requirement we discussed about in the exploratory dialog, but I don't think is stated in the digested version that is in this dialog. The disclosure code at every level of disclosure is correct Python that must and can be run without issues.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — that requirement should be explicit, and it changes how we think about disclosure.

I’d separate two invariants:

1. **Syntactic correctness**: disclosed code parses as valid Python.
2. **Execution correctness**: disclosed code can be executed in a reasonable namespace without raising.

If the requirement is truly execution correctness, then current `disclose` is probably only *partially* compliant. Some cases are safe:

```python
def foo(x:int) -> int: ...
```

But some disclosed skeletons can fail at execution time:

```python
@patch
def f(self:SomeClass): ...
```

This requires `patch` and `SomeClass` to exist.

Similarly:

```python
class Foo(Base): ...
```

requires `Base`.

And level 1 currently strips imports, so a level-1 disclosed cell may be syntactically valid but not executable in isolation.

So I think we should state the requirement very carefully:

> Disclosed code should be valid Python. For execution correctness, disclosed code is expected to run in the same dependency context as the selected dialog slice, unless exported as an isolated reuse bundle.

That gives us two modes:

- **digest / handoff / discovery**: parseable and structurally faithful; may depend on surrounding context
- **reuse / isolated bundle**: executable in a fresh namespace, with required imports/dependencies included

If you want the stronger invariant for *all* disclosure levels, then we need to adjust `disclose` to preserve enough imports/dependencies at every level, or inject safe placeholders for unresolved names. I’d be cautious with placeholders because they can make the code “run” while lying about actual dependencies.

The immediate next step I’d suggest is adding tiny helpers/tests that make the distinction visible:

```python
def compiles(src): return compile(src, '<disclosed>', 'exec')

def execs(src, ns=None):
    ns = {} if ns is None else ns
    exec(compiles(src), ns)
    return ns
```

Then a few documentation tests:

```python
src = 'def foo(x:int) -> int: return x+1'
for lvl in (1,2,3): execs(disclose(src, lvl))

src = 'from pathlib import Path\n\ndef foo(p:Path): return p'
execs(disclose(src, 2))
```

And one expected failure or design-test for level 1:

```python
src = 'from pathlib import Path\n\ndef foo(p:Path): return p'
test_fail(lambda: execs(disclose(src, 1)), contains='Path')
```

That failure is useful: it forces us to decide whether level 1 should preserve imports, use future annotations, remove annotations, or be documented as parseable-not-standalone-executable.

My preference: **don’t promise standalone execution for every digest profile**. Promise valid Python everywhere, and reserve standalone execution for a later “reuse/isolate” path. Otherwise the disclosure helper gets tangled with dependency tracking, which is a separate and harder problem.

<details class='token-usage-details' markdown='1'><summary>$0.0102</summary>

`total=63,723 | in=62,578 | out=1,145 | cached=3.9% | reasoning=516 | searches=0 | $0.0102`

</details>


Re: reuse/isolate path. I already have facilities in place to get partial and full isolation. We'll discuss that later or in another session.

For now, it's responsability of the user to mark the exported messages, including import and private helpers. I just wanted to state the requirement because I noted is not explicit anywhere in the code disclosure section.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that’s the right distinction:

- `disclose` should produce **valid runnable Python** at each disclosure level.
- It does **not** have to make an arbitrary selected subset isolated or dependency-complete.
- Dependency completeness is the responsibility of the selected/exported message set: imports, private helpers, base classes, decorators, etc. must be included when needed.

I’d add a short note to the code disclosure rationale section, something like:

```markdown
**Execution correctness requirement:** disclosed code at every level should be valid Python that can be executed in the appropriate dialog/export context. Disclosure is not responsible for dependency isolation: callers must include the necessary import cells, private helpers, decorators, base classes, and other dependencies in the selected/exported message set. Full or partial isolation is handled separately by isolation tooling.
```

That captures the invariant without overpromising standalone execution.

<details class='token-usage-details' markdown='1'><summary>$0.0098</summary>

`total=63,676 | in=63,336 | out=340 | cached=3.8% | reasoning=160 | searches=0 | $0.0098`

</details>


## chunking
Chunking is a deterministic inspection/materialization aid, not a robust mechanism for keeping over-budget context available to an LLM across turns.

Let's restart section splitting with the smallest deterministic piece: split a message list into contiguous section chunks. No budget logic yet.

```python
def section_chunks(msgs, level=2):
    "Split msgs into contiguous chunks starting at headers up to `level`."
    res,cur = [],None
    for m in msgs:
        lvl = _header_level(m)
        if lvl and lvl <= level:
            if cur: res.append(cur)
            cur = AD(title=m.content.splitlines()[0], msgs=[m])
        else:
            if cur is None: cur = AD(title='', msgs=[])
            cur.msgs.append(m)
    if cur: res.append(cur)
    return res
```

```python
syn = [mk('pre', 'x=1', 'code'), mk('h1', '## A'), mk('a', 'x=1', 'code'), mk('h2', '### A1'), mk('b', 'x=2', 'code'), mk('h3', '## B')]
chunks = section_chunks(syn)
test_eq([c.title for c in chunks], ['', '## A', '## B'])
test_eq([[m.id for m in c.msgs] for c in chunks], [['pre'], ['h1', 'a', 'h2', 'b'], ['h3']])
[[c.title, [m.id for m in c.msgs]] for c in chunks]
```

Hmmm, let's discuss first rationale and algorithm. I don't quite understand where do you want to go.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good call. I jumped too quickly into implementation.

The reason I suggested section splitting is this: once we have deterministic disclosure profiles, the next hard case is:

> “The smallest acceptable profile still doesn’t fit the token budget.”

At that point, if we refuse truncation and don’t want LLM summarization, the only deterministic option is to split the digest into multiple complete parts.

The question is: split by what?

The natural unit is **sections**, because the dialog is already structured as a document. A section chunk keeps related notes/code/tests together better than arbitrary message-count or token-count slicing.

So the rough pipeline would be:

1. Select messages with `filter_msgs_v2`
2. Apply a disclosure profile
3. Serialize and count tokens
4. If it fits, done
5. If it doesn’t fit, split selected messages into section chunks
6. Export each chunk separately, each with its own frontmatter
7. Optionally export a manifest listing all chunks

The algorithm I had in mind for splitting:

- Choose a section level, probably `##` by default
- Walk messages in dialog order
- Start a new chunk every time we hit a header at that level or above
- Include all nested subsections inside the parent chunk
- Preserve all message order
- Preserve complete messages
- Do not truncate content

Example:

```text
# dialog
## message utils
### message tag
### notes disclosure
## dialog utils
### dialog path
### dialog export
## distill
### filter messages v1
### filter messages v2
```

If splitting at level 2, chunks would be:

```text
chunk 1: # dialog preamble, maybe
chunk 2: ## message utils + all its ### subsections
chunk 3: ## dialog utils + all its ### subsections
chunk 4: ## distill + all its ### subsections
```

Why this helps:

- each chunk is semantically meaningful
- the consuming AI can understand one part at a time
- no message is dropped
- no content is truncated
- if a chunk is still too large, we can split at the next level, e.g. `###`

So it becomes a deterministic fallback ladder:

```text
try profile archive
try profile reuse
try profile handoff
try profile compact
try profile discovery
if still too big:
    split by ##
if chunks still too big:
    split by ###
if chunks still too big:
    split by individual messages
```

The “manifest” is important because it prevents split digests from becoming disorienting. Something like:

```text
Digest split into 4 parts:
1. ## message utils — 18 messages — 1200 tokens
2. ## dialog utils — 24 messages — 1800 tokens
3. ## Code message disclosure — 31 messages — 2400 tokens
4. ## distill — 26 messages — 2100 tokens
```

My main rationale is: **budget management should degrade by lowering disclosure first, then by splitting semantically, never by truncating.**

But we don’t need to implement all that yet. The first conceptual decision is whether `##` sections are the right default chunking unit, or whether the split level should depend on the selected messages/profile/budget.

<details class='token-usage-details' markdown='1'><summary>$0.0102</summary>

`total=64,760 | in=64,033 | out=727 | cached=3.8% | reasoning=42 | searches=0 | $0.0102`

</details>


And the workflow then would be one tool call that returns full digest if in budget or a list of chunks ids + what? if not, that the AI can request in succesive tool calls? If that's the case then we could be tempted to introduce non-deterministic summarization, if you have other idea, let's see it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that’s exactly the fork in the design:

- If disclosed digest fits budget: return the digest.
- If not: return a **deterministic manifest**, not a summary.

The manifest should be a navigable table of contents, not an LLM-written compaction. It can contain only things we can compute exactly from the dialog:

- `digest_id`
- chosen selection query
- source dialog path
- total selected messages
- profiles tried + token counts
- split strategy used, e.g. `section_level=2`
- chunk ids
- section title / section path
- message ids in each chunk
- message count
- token counts for each profile or selected profile
- whether chunk fits the requested budget
- possibly first/last message ids

So a not-in-budget result is more like:

```text
Digest does not fit 4000 tokens.

Smallest profile:
- profile: discovery
- tokens: 8500
- split_needed: true

Available chunks:
1. chunk_id: c1
   title: ## message utils
   msgs: 18
   tokens: {discovery: 700, handoff: 1300, archive: 2800}

2. chunk_id: c2
   title: ## dialog utils
   msgs: 24
   tokens: {discovery: 900, handoff: 1600, archive: 3200}

3. chunk_id: c3
   title: ## distill
   msgs: 42
   tokens: {discovery: 1500, handoff: 3100, archive: 7600}
   split_needed: true
```

Then the AI can request:

```text
give me chunk c2 at handoff profile
```

or:

```text
split chunk c3 one level deeper
```

No nondeterministic summarization needed.

I’d think of this as **pagination over structured dialog context**, not summarization. The manifest is like an index page. The chunks are exact disclosed exports. If a chunk still doesn’t fit, split deeper. If individual messages still don’t fit at lowest disclosure, that’s an explicit failure case:

```text
message _abc123 exceeds budget even at discovery profile
```

At that point, under the “no truncation / no LLM compaction / no message loss” rules, there is no valid digest answer. The system must say so.

The key design is that the manifest itself can use deterministic disclosure too:

- section title from header line
- message ids
- message types
- metadata flags
- token counts
- maybe `clamp_note(..., 'header')` for section notes
- maybe code skeleton count, but not prose summary

So instead of:

> “This section discusses message utilities…”

we use:

```text
## message utils
messages: 31
types: note=8, code=23
metadata: exported=11, echo=4, hidden=2
subsections:
- ### message tag
- ### message metadata & directives
- ### notes disclosure
```

That’s deterministic and useful.

I see three useful return modes:

1. **`digest`**
   - full requested digest fits
   - return text directly

2. **`manifest`**
   - full digest doesn’t fit
   - return deterministic chunk index
   - caller can request chunks by id

3. **`package`**
   - for non-interactive workflows
   - write chunks to linked messages/files
   - return a manifest with links/ids to all parts

The third mode is important in Solveit. If we can materialize all chunks as note messages or files, then one call can create the whole split package without trying to fit all content into the model response. The AI/user sees the manifest and follows links as needed.

So the robust workflow is:

```text
make_digest(...)
    if fits: return digest
    else:
        create chunk records/messages/files
        return manifest
```

The manifest can say:

```text
part_001 -> linked note/message id _abc
part_002 -> linked note/message id _def
part_003 -> linked note/message id _ghi
```

That avoids needing nondeterministic summaries and avoids stuffing all chunks into one response.

I’d avoid automatic “smart summary” entirely in this layer. If later we want LLM summaries, they should be a separate, explicitly lossy layer with frontmatter like:

```yaml
lossy_summary: true
summary_model: ...
source_digest_id: ...
```

But the core digest system should stay deterministic:

> select → disclose → serialize → count → fit or split → manifest.

<details class='token-usage-details' markdown='1'><summary>$0.0108</summary>

`total=66,329 | in=64,842 | out=1,487 | cached=3.8% | reasoning=516 | searches=0 | $0.0108`

</details>


Looks good (and hard). The only complex point I see is that splitting the digest in chunks implies adding state between calls. We should be very careful with that and move around an object or a proxy. Remember also that digest discovery can be requested on running dialogs (in memory, with a runnign kernel) or on stored dialog in disk (.ipynb files). In any case, the digest system must maintain state between call to be able to generate and return the chunks. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — state is the hard part, and I think we should treat it as a first-class design constraint, not an implementation detail.

The main design decision I’d make:

> A digest discovery call should create a **digest plan**, and chunk retrieval should operate on that plan, not re-run the query against a changing dialog.

Otherwise a running dialog can change between calls, and `chunk_003` might no longer mean the same thing.

So the core object is something like:

```text
DigestPlan
- id
- source
- created_at
- selection/query metadata
- selected message snapshot or stable source refs
- disclosure profiles tried
- chosen profile or split-needed state
- chunks
- manifest
```

The important question is whether the plan stores **copies of messages** or only **references to message ids**.

My take:

- For correctness and reproducibility, default should be **snapshot/copy selected messages into the plan**.
- For lightweight/live workflows, we can later allow `live=True`, where chunks resolve ids from the current dialog at retrieval time.
- But deterministic digest behavior should be snapshot-based.

That gives us two modes:

```text
snapshot mode:
    chunk retrieval returns exactly what discovery saw

live mode:
    chunk retrieval reflects current dialog state
```

For AI context and handoff, snapshot mode is safer.

For implementation, I’d separate:

1. **Source adapter**
   - running dialog adapter: `find_msgs_x(...)`
   - notebook-on-disk adapter: read `.ipynb`, convert cells/messages to `MsgX`

2. **Plan builder**
   - receives `list[MsgX]`
   - filters/scopes
   - computes profile sizes
   - computes chunk boundaries
   - stores selected message snapshot

3. **Plan store**
   - in-memory registry for current session
   - optionally durable file-backed store later

4. **Chunk renderer**
   - `render_digest(plan, profile=...)`
   - `render_chunk(plan, chunk_id, profile=...)`
   - `render_manifest(plan)`

The source adapter is the only place that cares whether the dialog is running or stored on disk. After that, everything works from `list[MsgX]`.

I’d be wary of relying only on a global in-memory dict, because:

- kernel restart loses it
- handoff/fork wants durable references
- a digest may outlive the current interaction

But in-memory is a perfectly good first implementation if the plan object itself can later be serialized.

Something like this conceptual workflow:

```text
plan = make_digest_plan(...)
if plan.fits:
    return render_digest(plan)
else:
    return render_manifest(plan)

later:
    return render_chunk(plan, 'chunk_003')
```

In Solveit specifically, there are three plausible state carriers:

1. **Python object**
   - easiest during one running session
   - e.g. `plan = make_digest_plan(...)`

2. **Registry/proxy id**
   - `digest_id -> plan`
   - tool/API returns `digest_id`, later calls pass it back

3. **Linked message/file**
   - durable, visible, forkable
   - manifest can point to generated note/file ids

I think the clean architecture is: **plan object first, registry second, durable materialization third**.

So internally everything takes a plan object. The registry is just convenience:

```text
_digest_plans[plan.id] = plan
```

And durable storage is another adapter:

```text
save_digest_plan(plan)
load_digest_plan(id)
```

For running dialogs, I’d also store source fingerprints in the plan:

```text
source:
  kind: running_dialog
  path: /prj/pote/nbs/01_dialog
  msg_count: 240
  first_msg_id: ...
  last_msg_id: ...
```

For disk dialogs:

```text
source:
  kind: notebook_file
  path: ...
  mtime: ...
  sha256: ...
```

Even if chunks use a snapshot, these fingerprints help explain where the plan came from and detect drift if someone tries live resolution later.

The trickiest edge case is chunk storage. If the plan stores full copied messages, then retrieval is deterministic but the plan may be large. But that’s acceptable because the whole reason we’re doing this is that the full digest may be too large for one context. State can be larger than one model response.

If we don’t want to store full message copies, we can store:

```text
selected_ids
source_fingerprint
```

and retrieve by ids later. But then we must accept drift or check the fingerprint and refuse if changed. I’d rather not start there.

So my preferred invariant:

> A `DigestPlan` is a frozen selected message set plus deterministic rendering metadata.

Then chunk ids are stable because they refer to the plan’s own frozen message list, not the external dialog.

The manifest should probably include:

```yaml
digest_id: ...
state: snapshot
source:
  kind: running_dialog | notebook_file
  path: ...
profile_order: [...]
requested_budget: ...
smallest_profile_tokens: ...
chunks:
  - id: chunk_001
    title: "## message utils"
    msg_count: 18
    msg_ids: [...]
    tokens:
      discovery: 700
      handoff: 1300
      archive: 2800
```

The user/AI can then request chunk ids against that digest id.

I’d postpone lossy summarization entirely. The stateful plan/chunk system is harder, but it keeps the core promise:

> deterministic, reproducible, non-lossy-within-disclosure context generation.

<details class='token-usage-details' markdown='1'><summary>$0.0111</summary>

`total=67,537 | in=65,971 | out=1,566 | cached=3.7% | reasoning=392 | searches=0 | $0.0111`

</details>


Ok, that's a lot of words. Let's simplify. We're in Solveit where eveything run in a stateful kernel. I see two scenarios:
- User driven disclosure. The user request a digest and depending of the goal and the return can easily request additional chunks. No context issues here, only pure python. For instance, the user want to include in a dialog the digest to discuss it with the model. Is the user who determines what becomes context by including whatever they want in the dialog.
- LLM calls the digest tool with an explicit budget. The function call returns in-budget string, good, continue. If not, it returns a manifest that should be also a string (or a simple object easily serializable as string, and not all models (or harnesses) can handle non-string returns). Remember that solveit (and many other frameworks) truncate tool responses at 2k. So, those 2k max chars are what will become context in the succesive tool calls, not the real tool response. Even in the case there's not manifest, the AI can make whatever it want with the response only in the turn the tool is called, in the next tool calls what becomes context is outside their control.

Though I want this to be a general utility usable outside of Solveit, let's get first a MVP solution here in Solveit and refine later. The function doing the digest run in in the same namespace that the dialog and can store whatever it need in the user ns and in variables that the model can always access (with the $`` magic). The manifest should be the minimun possible to allow the receiver to determine what to call next. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — this simplifies the design a lot.

For the MVP, I’d avoid durable files, proxy objects, source adapters, and generic framework concerns. In Solveit we can rely on the stateful Python namespace.

The core idea:

- the digest function stores full state in a global Python dict
- the tool/function return is always a string
- if full digest fits, return the digest string
- if not, return a **tiny manifest string**
- the manifest only needs enough information for the model/user to request the next chunk
- all heavy data stays in `_digest_plans`

Something like:

```python
_digest_plans = {}
```

Then a plan contains:

```python
AD(
    id='dg_xxx',
    msgs=selected_msgs,
    chunks=chunks,
    profiles=profile_rows,
    fmt='md',
    frontmatter=frontmatter,
)
```

The manifest should not try to be complete context. It should be a routing card:

```text
DIGEST MANIFEST
id: dg_abc123
budget: 2000
selected_msgs: 84
smallest_profile: discovery
smallest_tokens: 2671
status: split_needed

chunks:
0  ## message utils                 21 msgs  680 tok
1  ## dialog utils                  18 msgs  520 tok
2  ## Code message disclosure       24 msgs  910 tok
3  ## distill                       21 msgs  760 tok

next:
digest_chunk('dg_abc123', 0)
digest_chunk('dg_abc123', 1)
digest_chunk('dg_abc123', 2)
digest_chunk('dg_abc123', 3)
```

No message ids, no full profile matrix, no prose summary. Just enough to continue.

For LLM-tool usage, the next calls are deterministic:

```python
digest_chunk('dg_abc123', 2)
```

For user-driven use, the user can inspect variables directly:

```python
_digest_plans['dg_abc123']
```

or export/link chunks manually.

I’d keep the MVP API to just two public helpers:

```python
digest_run(...)
digest_chunk(digest_id, chunk_id, profile=None)
```

Maybe one inspection helper later:

```python
digest_manifest(digest_id)
```

The important invariant:

> The manifest is not the context. The stored plan is the state. The manifest is only a compact pointer into that state.

That fits Solveit well because `$` variable access can expose `_digest_plans` when needed, while normal tool calls can stay string-only and under truncation limits.

I’d also make the digest id short and human-readable, e.g.:

```python
dg_001
dg_002
```

instead of UUIDs. In a stateful session, uniqueness across the current namespace is enough.

MVP flow:

1. `digest_run(...)`
   - select messages
   - try profiles against budget
   - if one fits, return digest
   - else build chunks, store plan, return tiny manifest

2. `digest_chunk(did, i, profile=None)`
   - load plan from `_digest_plans`
   - render chunk `i`
   - if chunk still too large at requested profile, return a smaller manifest for subchunks later

I’d postpone recursive subchunking until we actually hit that case. First MVP can simply say:

```text
chunk 2 still exceeds budget at discovery profile; split_level=3 needed
```

Then implement deeper splitting only when needed.

So the next implementation step should be small: create `_digest_plans`, a digest id helper, and a compact manifest renderer over a synthetic list of chunks.

<details class='token-usage-details' markdown='1'><summary>$0.0111</summary>

`total=68,824 | in=67,541 | out=1,283 | cached=3.6% | reasoning=516 | searches=0 | $0.0111`

</details>


Let's start with the tiny state + manifest layer.

In [ ]:
#| export
_digest_plans,_digest_seq = {},0

def _new_digest_id():
    global _digest_seq
    _digest_seq += 1
    return f'dg_{_digest_seq:03d}'

In [ ]:
#| export
def _chunk_tok(c, profile='discovery'): return c.tokens.get(profile, c.tokens[min(c.tokens, key=c.tokens.get)]) if isinstance(c.tokens, dict) else c.tokens
def _chunk_title(c, i): return c.title or f'chunk {i}'

In [ ]:
def digest_manifest_txt(p, profile='discovery'):
    "Return compact string manifest for digest plan `p`."
    lines = [
        f'DIGEST MANIFEST {p.id}',
        f'budget: {p.budget}',
        f'selected_msgs: {len(p.msgs)}',
        f'status: {p.status}',
        f'profile: {profile}',
        '',
        'chunks:']
    for i,c in enumerate(p.chunks): lines.append(f'{i}: {_chunk_title(c, i)} | {len(c.msgs)} msgs | {_chunk_tok(c, profile)} tok | digest_chunk({p.id!r}, {i})')
    return '\n'.join(lines)

In [ ]:
syn_chunks = [
    AD(title='## message utils', msgs=[mk('a'), mk('b')], tokens=dict(discovery=120, handoff=260)),
    AD(title='## dialog utils', msgs=[mk('c')], tokens=dict(discovery=80, handoff=180))
]
p = AD(id=_new_digest_id(), budget=500, msgs=[m for c in syn_chunks for m in c.msgs], status='split_needed', chunks=syn_chunks)
_digest_plans[p.id] = p

txt = digest_manifest_txt(p)
test_is(p.id in _digest_plans, True)
test_is('digest_chunk' in txt, True)
print(txt)

DIGEST MANIFEST dg_001
budget: 500
selected_msgs: 3
status: split_needed
profile: discovery

chunks:
0: ## message utils | 2 msgs | 120 tok | digest_chunk('dg_001', 0)
1: ## dialog utils | 1 msgs | 80 tok | digest_chunk('dg_001', 1)


Next smallest useful piece: create real contiguous chunks from a message list. This gives the manifest something concrete to point at, still with no budget logic.

In [ ]:
def section_chunks(msgs, level=2):
    "Split msgs into contiguous chunks starting at headers up to `level`."
    res,cur = [],None
    for m in msgs:
        lvl = _header_level(m)
        if lvl and lvl <= level:
            if cur: res.append(cur)
            cur = AD(title=m.content.splitlines()[0], msgs=[m])
        else:
            if cur is None: cur = AD(title='', msgs=[])
            cur.msgs.append(m)
    if cur: res.append(cur)
    return res

Tiny check:

In [ ]:
syn = [mk('pre', 'x=1', 'code'), mk('h1', '## A'), mk('a', 'x=1', 'code'), mk('h2', '### A1'), mk('b', 'x=2', 'code'), mk('h3', '## B')]
chunks = section_chunks(syn)

test_eq([c.title for c in chunks], ['', '## A', '## B'])
test_eq([[m.id for m in c.msgs] for c in chunks], [['pre'], ['h1', 'a', 'h2', 'b'], ['h3']])

[[c.title, [m.id for m in c.msgs]] for c in chunks]

[['', ['pre']], ['## A', ['h1', 'a', 'h2', 'b']], ['## B', ['h3']]]

Next smallest piece: annotate chunks with token counts per profile, so the existing manifest can report real numbers.

In [ ]:
def chunk_tokens(c, profiles=('discovery','compact','handoff','reuse','archive'), fmt='md'):
    "Return token counts for chunk `c` across profiles."
    return {p:count_tokens(exportmsgs_profile(c.msgs, p, fmt=fmt)) for p in profiles}

def add_chunk_tokens(chunks, profiles=('discovery','compact','handoff','reuse','archive'), fmt='md'):
    "Add profile token counts to chunks."
    for c in chunks: c.tokens = chunk_tokens(c, profiles, fmt)
    return chunks

Tiny check:

In [ ]:
chunks = add_chunk_tokens(section_chunks(syn), profiles=('discovery','handoff'))
p = AD(id=_new_digest_id(), budget=500, msgs=syn, status='split_needed', chunks=chunks)
_digest_plans[p.id] = p

txt = digest_manifest_txt(p, profile='discovery')
test_is('digest_chunk' in txt, True)
test_is('## A' in txt, True)
print(txt)

DIGEST MANIFEST dg_002
budget: 500
selected_msgs: 6
status: split_needed
profile: discovery

chunks:
0: chunk 0 | 1 msgs | 14 tok | digest_chunk('dg_002', 0)
1: ## A | 4 msgs | 47 tok | digest_chunk('dg_002', 1)
2: ## B | 1 msgs | 9 tok | digest_chunk('dg_002', 2)


Next, let’s make the manifest’s `digest_chunk(...)` calls real: render one stored chunk from `_digest_plans`.

In [ ]:
def digest_chunk(digest_id, i, profile='discovery', fmt='md', **kwargs):
    "Return rendered chunk `i` from stored digest plan."
    p = _digest_plans[digest_id]
    c = p.chunks[i]
    fm = dict(path=getattr(p, 'path', ''), description=f'{p.id} chunk {i}: {_chunk_title(c, i)}', n_msgs=len(c.msgs), fmt=fmt)
    return exportmsgs_profile(c.msgs, profile, fmt=fmt, frontmatter=fm, **kwargs)

Tiny check:

In [ ]:
txt = digest_chunk(p.id, 1, profile='discovery')
test_is('## A' in txt, True)
test_is(f'{p.id} chunk 1' in txt, True)
print(txt)

---
digest: solveit-dialog
path: 
exported_at: 2026-05-18T12:09:50+00:00
description: 'dg_002 chunk 1: ## A'
profile: 'discovery'
disclosure: {"note_level": "header", "code_level": 1, "output": false}
docs: 'dialog_digest_docs()'
stats: {total: 4, exported: 4}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #h1 note -->
## A

<!-- #a code -->
```python
x=1
```

<!-- #h2 note -->
### A1

<!-- #b code -->
```python
x=2
```



Next small piece: a first `digest_run` that either returns the fitted digest or stores a chunked plan and returns the compact manifest.

In [ ]:
def digest_run(msgs:list[MsgX], max_tokens:int, profiles=None, fmt='md', split_level=2, path='', description='dialog digest', **kwargs):
    "Return in-budget digest or store chunk plan and return compact manifest."
    profiles = profiles or _profile_order
    fm = dict(path=path, description=description, n_msgs=len(msgs), fmt=fmt)
    r = fit_digest(msgs, max_tokens, profiles=profiles, fmt=fmt, frontmatter=fm, **kwargs)
    if r.profile: return r.text
    chunks = add_chunk_tokens(section_chunks(msgs, split_level), profiles=profiles, fmt=fmt)
    p = AD(id=_new_digest_id(), budget=max_tokens, msgs=msgs, status='split_needed', chunks=chunks, profiles=profiles, fmt=fmt, path=path, description=description)
    _digest_plans[p.id] = p
    return digest_manifest_txt(p, profile=profiles[-1])

Tiny check:

In [ ]:
txt = digest_run(syn, 10000, profiles=['handoff', 'discovery'], description='fits test')
test_is('DIGEST MANIFEST' in txt, False)
test_is('## A' in txt, True)

txt = digest_run(syn, 1, profiles=['handoff', 'discovery'], description='split test')
test_is('DIGEST MANIFEST' in txt, True)
test_is('digest_chunk' in txt, True)
print(txt)

DIGEST MANIFEST dg_004
budget: 1
selected_msgs: 6
status: split_needed
profile: discovery

chunks:
0: chunk 0 | 1 msgs | 14 tok | digest_chunk('dg_004', 0)
1: ## A | 4 msgs | 47 tok | digest_chunk('dg_004', 1)
2: ## B | 1 msgs | 9 tok | digest_chunk('dg_004', 2)


Next I’d smoke-test the MVP on the real selected message set: force a split, confirm a plan is stored, then render one chunk from the manifest.

In [ ]:
txt = digest_run(v2, 1000, profiles=['handoff', 'compact', 'discovery'], path=find_dname(), description='v2 digest split smoke test')
did = f'dg_{_digest_seq:03d}'
test_is('DIGEST MANIFEST' in txt, True)
test_is(did in _digest_plans, True)
print(txt)

DIGEST MANIFEST dg_007
budget: 1000
selected_msgs: 84
status: split_needed
profile: discovery

chunks:
0: ## message utils | 16 msgs | 489 tok | digest_chunk('dg_007', 0)
1: ## dialog utils | 21 msgs | 445 tok | digest_chunk('dg_007', 1)
2: ## Code message disclosure | 13 msgs | 437 tok | digest_chunk('dg_007', 2)
3: ## distill | 34 msgs | 1174 tok | digest_chunk('dg_007', 3)


Then render one chunk:

In [ ]:
chunk = digest_chunk(did, 0, profile='discovery')
test_is('profile: ' in chunk, True)
print(chunk[:2000])

---
digest: solveit-dialog
path: /prj/pote/nbs/01_dialog
exported_at: 2026-05-18T12:13:56+00:00
description: 'dg_006 chunk 0: ## message utils'
profile: 'discovery'
disclosure: {"note_level": "header", "code_level": 1, "output": false}
docs: 'dialog_digest_docs()'
stats: {total: 16, exported: 16}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_e069f418 note -->
## message utils

<!-- #_ea3120ed note -->
### message tag

<!-- #_e2f454e8 code -->
```python
Msg: TypeAlias = AD
```

<!-- #_7d419edc note -->
### message metadata & directives

<!-- #_084ce165 code -->
```python
_exports = {'exporti', 'exports', 'export'}

_hidden = {'hide'}

_echo = {'echo', 'echo:'}
```

<!-- #_8e71d4cb code -->
```python
def is_exported(m: Msg) -> bool: ...

def is_hidden(m: Msg) -> bool: ...

def is_echo(m: Msg) -> bool: ...

def is_header(m: Msg) -> bool: ...

def is_skipped(m: Msg) -> bool: ...


The smoke test surfaced a useful issue: some disclosed code cells become empty, so the markdown serializer emits blank code fences. Since we want message coverage and runnable disclosed code, a good small patch is to make empty disclosed code become `...`.

In [ ]:
def disclose_msg(m:MsgX, note_level='header++', code_level=2, output=False) -> MsgX:
    "Return a disclosed copy of message `m`."
    res = msgx(AD(m.copy()))
    if res.msg_type == 'note' and note_level != 'full': res.content = clamp_note(res.content, note_level)
    if res.msg_type == 'code' and code_level != 'full':
        res.content = disclose(strip_directives(res.content), code_level)
        if not res.content: res.content = '...'
    if not output: res.output = ''
    return res

Tiny check:

In [ ]:
m = mk('t', 'test_eq(1, 1)', 'code')
d = disclose_msg(m, code_level=1)
test_eq(d.content, '...')
test_is(m.content == 'test_eq(1, 1)', True)
d.content

'...'

In [ ]:
txt = digest_run(v2, 1000, profiles=['handoff', 'compact', 'discovery'], path=find_dname(), description='v2 digest split smoke test')
did = f'dg_{_digest_seq:03d}'
test_is('DIGEST MANIFEST' in txt, True)
test_is(did in _digest_plans, True)
print(txt)

DIGEST MANIFEST dg_008
budget: 1000
selected_msgs: 84
status: split_needed
profile: discovery

chunks:
0: ## message utils | 16 msgs | 489 tok | digest_chunk('dg_008', 0)
1: ## dialog utils | 21 msgs | 445 tok | digest_chunk('dg_008', 1)
2: ## Code message disclosure | 13 msgs | 437 tok | digest_chunk('dg_008', 2)
3: ## distill | 34 msgs | 1174 tok | digest_chunk('dg_008', 3)


In [ ]:
chunk = digest_chunk(did, 0, profile='discovery')
test_is('profile: ' in chunk, True)
print(chunk[:2000])

---
digest: solveit-dialog
path: /prj/pote/nbs/01_dialog
exported_at: 2026-05-18T12:17:55+00:00
description: 'dg_008 chunk 0: ## message utils'
profile: 'discovery'
disclosure: {"note_level": "header", "code_level": 1, "output": false}
docs: 'dialog_digest_docs()'
stats: {total: 16, exported: 16}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_e069f418 note -->
## message utils

<!-- #_ea3120ed note -->
### message tag

<!-- #_e2f454e8 code -->
```python
Msg: TypeAlias = AD
```

<!-- #_7d419edc note -->
### message metadata & directives

<!-- #_084ce165 code -->
```python
_exports = {'exporti', 'exports', 'export'}

_hidden = {'hide'}

_echo = {'echo', 'echo:'}
```

<!-- #_8e71d4cb code -->
```python
def is_exported(m: Msg) -> bool: ...

def is_hidden(m: Msg) -> bool: ...

def is_echo(m: Msg) -> bool: ...

def is_header(m: Msg) -> bool: ...

def is_skipped(m: Msg) -> bool: ...


The remaining blank fence is likely whitespace-only disclosed code, so the guard should use `.strip()`:

In [ ]:
#| export
def disclose_msg(m:MsgX, note_level='header++', code_level=2, output=False) -> MsgX:
    "Return a disclosed copy of message `m`."
    res = msgx(AD(m.copy()))
    if res.msg_type == 'note' and note_level != 'full': res.content = clamp_note(res.content, note_level)
    if res.msg_type == 'code' and code_level != 'full':
        res.content = disclose(strip_directives(res.content), code_level)
        if not res.content.strip(): res.content = '...'
    if not output: res.output = ''
    return res

Tiny check against the case we just saw:

In [ ]:
m = mk('cls', 'class MsgX(Msg): ...', 'code')
d = disclose_msg(m, code_level=1)
test_eq(d.content, '...')
d.content

'...'

In [ ]:
txt = digest_run(v2, 1000, profiles=['handoff', 'compact', 'discovery'], path=find_dname(), description='v2 digest split smoke test')
did = f'dg_{_digest_seq:03d}'
test_is('DIGEST MANIFEST' in txt, True)
test_is(did in _digest_plans, True)
print(txt)

DIGEST MANIFEST dg_009
budget: 1000
selected_msgs: 84
status: split_needed
profile: discovery

chunks:
0: ## message utils | 16 msgs | 490 tok | digest_chunk('dg_009', 0)
1: ## dialog utils | 21 msgs | 445 tok | digest_chunk('dg_009', 1)
2: ## Code message disclosure | 13 msgs | 437 tok | digest_chunk('dg_009', 2)
3: ## distill | 34 msgs | 1174 tok | digest_chunk('dg_009', 3)


In [ ]:
chunk = digest_chunk(did, 0, profile='discovery')
test_is('profile: ' in chunk, True)
print(chunk[:2000])

---
digest: solveit-dialog
path: /prj/pote/nbs/01_dialog
exported_at: 2026-05-18T12:20:42+00:00
description: 'dg_009 chunk 0: ## message utils'
profile: 'discovery'
disclosure: {"note_level": "header", "code_level": 1, "output": false}
docs: 'dialog_digest_docs()'
stats: {total: 16, exported: 16}
conventions: '<!-- #<msg_id> <msg_type> --> precedes each block; ```output blocks contain cell output; ```response blocks contain model response'
---

<!-- #_e069f418 note -->
## message utils

<!-- #_ea3120ed note -->
### message tag

<!-- #_e2f454e8 code -->
```python
Msg: TypeAlias = AD
```

<!-- #_7d419edc note -->
### message metadata & directives

<!-- #_084ce165 code -->
```python
_exports = {'exporti', 'exports', 'export'}

_hidden = {'hide'}

_echo = {'echo', 'echo:'}
```

<!-- #_8e71d4cb code -->
```python
def is_exported(m: Msg) -> bool: ...

def is_hidden(m: Msg) -> bool: ...

def is_echo(m: Msg) -> bool: ...

def is_header(m: Msg) -> bool: ...

def is_skipped(m: Msg) -> bool: ...


The next smallest useful step is to make the manifest show which chunks fit the requested budget. The smoke test already surfaced `## distill` as over budget, so the manifest should say that explicitly.

In [ ]:
def digest_manifest_txt(p, profile='discovery'):
    "Return compact string manifest for digest plan `p`."
    lines = [
        f'DIGEST MANIFEST {p.id}',
        f'budget: {p.budget}',
        f'selected_msgs: {len(p.msgs)}',
        f'status: {p.status}',
        f'profile: {profile}',
        '',
        'chunks:']
    for i,c in enumerate(p.chunks):
        toks = _chunk_tok(c, profile)
        stat = 'ok' if toks <= p.budget else 'over'
        lines.append(f'{i}: {stat} | {_chunk_title(c, i)} | {len(c.msgs)} msgs | {toks} tok | digest_chunk({p.id!r}, {i})')
    return '\n'.join(lines)

Tiny check against the current real plan:

In [ ]:
txt = digest_manifest_txt(_digest_plans[did])
test_is('over | ## distill' in txt, True)
test_is('ok | ## message utils' in txt, True)
print(txt)

DIGEST MANIFEST dg_009
budget: 1000
selected_msgs: 84
status: split_needed
profile: discovery

chunks:
0: ok | ## message utils | 16 msgs | 490 tok | digest_chunk('dg_009', 0)
1: ok | ## dialog utils | 21 msgs | 445 tok | digest_chunk('dg_009', 1)
2: ok | ## Code message disclosure | 13 msgs | 437 tok | digest_chunk('dg_009', 2)
3: over | ## distill | 34 msgs | 1174 tok | digest_chunk('dg_009', 3)


Next small safety patch: make `digest_chunk` respect the stored budget by default. If a chunk is over budget, it returns a tiny deterministic notice instead of a too-large chunk. The user can still force it manually.

In [ ]:
#| export
def _over_chunk_txt(p, i, profile):
    c,toks = p.chunks[i],_chunk_tok(p.chunks[i], profile)
    return '\n'.join([
        f'DIGEST CHUNK OVER BUDGET {p.id}',
        f'chunk: {i}',
        f'title: {_chunk_title(c, i)}',
        f'budget: {p.budget}',
        f'tokens: {toks}',
        f'profile: {profile}',
        f'force: digest_chunk({p.id!r}, {i}, profile={profile!r}, force=True)'])

def digest_chunk(digest_id, i, profile='discovery', fmt='md', force=False, **kwargs):
    "Return rendered chunk `i` from stored digest plan."
    p = _digest_plans[digest_id]
    if not force and _chunk_tok(p.chunks[i], profile) > p.budget: return _over_chunk_txt(p, i, profile)
    c = p.chunks[i]
    fm = dict(path=getattr(p, 'path', ''), description=f'{p.id} chunk {i}: {_chunk_title(c, i)}', n_msgs=len(c.msgs), fmt=fmt)
    return exportmsgs_profile(c.msgs, profile, fmt=fmt, frontmatter=fm, **kwargs)

Tiny check against the current over-budget `## distill` chunk:

In [ ]:
txt = digest_chunk(did, 3)
test_is('DIGEST CHUNK OVER BUDGET' in txt, True)
test_is('force=True' in txt, True)

forced = digest_chunk(did, 3, force=True)
test_is('## distill' in forced, True)

print(txt)

DIGEST CHUNK OVER BUDGET dg_009
chunk: 3
title: ## distill
budget: 1000
tokens: 1174
profile: discovery
force: digest_chunk('dg_009', 3, profile='discovery', force=True)


Ok, are we finished here?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

For this MVP slice, yes — I think we have a coherent stopping point.

What’s working now:

- deterministic disclosure profiles: `discovery`, `compact`, `handoff`, `reuse`, `archive`
- message-level and list-level disclosure: `disclose_msg`, `disclose_msgs`, `disclose_profile`
- profile-aware export: `exportmsgs_profile`
- frontmatter records profile/disclosure/docs
- budget fitting: `fit_digest`
- stateful digest plans in `_digest_plans`
- compact manifest when a digest does not fit
- section-based chunks with token counts
- chunk retrieval via `digest_chunk`
- over-budget chunk guard with `force=True`

The current model is:

```text
select msgs -> try full digest -> if too large, store plan -> return tiny manifest -> request chunks
```

The main unfinished pieces I’d leave for later:

- recursive subchunking for chunks that are still over budget
- polishing/export marking across the whole dialog
- integrating isolation/reuse tooling
- possibly making plan storage durable outside the current kernel
- cleaning up names/signatures once the design settles

So yes: good checkpoint. The next useful action is probably not more implementation, but a sweep/cleanup pass over the dialog: mark exports, tighten notes, and decide which of these helpers are API vs exploratory.

<details class='token-usage-details' markdown='1'><summary>$0.0116</summary>

`total=76,208 | in=75,932 | out=276 | cached=89.5% | searches=0 | $0.0116`

</details>


I'm not convinced of chunking usefulness. For a human user writing the dialog, yes, they can decide what, when and how setup context. At the end, that's a dialog, frozen context.

For an AI, I don't visualize how the model could use our tools. Suppose we're in a conversation in other dialog, and I ask of you to create a messages, or messages, below with a digest of this dialog. How would you act?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I think your skepticism is right. For an AI, chunking is only useful if it **materializes context somewhere stable**. If it just returns a manifest and expects the model to “navigate” tool calls, it’s fragile and awkward.

If you asked me in another dialog:

> create message/messages below with a digest of this dialog

I would ideally act like this:

1. decide the purpose/profile, probably `handoff` unless you specified otherwise
2. select the source messages
3. generate the digest
4. if it fits comfortably, create **one note message** containing the digest
5. if it does not fit, create:
   - one short **index/manifest note**
   - one note per chunk/part
6. return only a short confirmation with the created message ids

The important point: I would not rely on the manifest becoming model context. The digest/chunks should be written into the dialog as messages. Then the dialog itself is the context.

So chunking is useful less as “AI navigation” and more as **dialog materialization**:

```text
[source dialog] -> digest package -> [target dialog messages]
```

A good created package might look like:

```text
## Digest: /prj/pote/nbs/01_dialog

Profile: handoff
Parts: 4

- Part 1: message utils
- Part 2: dialog utils
- Part 3: code disclosure
- Part 4: distill/query helpers
```

Then below it:

```text
## Digest part 1/4: message utils
...
```

```text
## Digest part 2/4: dialog utils
...
```

and so on.

That means the AI does not have to remember hidden tool state. It does not have to keep making follow-up calls. The output is now normal Solveit dialog content.

Where chunking helps:

- one huge note message may be unpleasant to edit/read
- separate parts can be skipped, moved, deleted, or rewritten
- the target dialog can include only the chunks the user cares about
- future model calls see ordinary messages, not ephemeral tool results

Where chunking is weak:

- if the tool only returns strings and those strings are truncated before the model can use them
- if the AI is expected to inspect chunk manifests and decide interactively without writing anything stable
- if chunks are over-budget and there is no materialization function that writes directly to the dialog

So I’d revise the design goal:

> chunking is not mainly for the AI to consume through tool responses; chunking is for creating a digest package as real dialog messages.

That suggests the more useful MVP API is not only:

```python
digest_run(...)
digest_chunk(...)
```

but eventually:

```python
digest_to_msgs(...)
```

or similar: a function that writes the full package into the current/target dialog and returns a tiny report.

Conceptually:

```python
async def digest_to_msgs(msgs, max_tokens=4000, profile='handoff', dname=None):
    ...
```

Return:

```text
Created digest package dg_012:
- index: _abc123
- part 1: _def456
- part 2: _ghi789
```

That avoids the core tool-response truncation problem because the big content is not returned to the model; it is written into the dialog.

So my current take:

- `digest_run` / `digest_chunk`: useful for debugging and user-driven exploration
- chunking manifest: useful as a compact routing/index object
- **real AI workflow needs materialization**: write digest/chunks as messages/files and return refs
- without materialization, chunking is much less compelling for AI-driven workflows

<details class='token-usage-details' markdown='1'><summary>$0.0124</summary>

`total=77,643 | in=76,356 | out=1,287 | cached=90.4% | reasoning=516 | searches=0 | $0.0124`

</details>


Yes, that's my take too. Chunking seems a good idea but for humans writers is only a hint to divide one big messages in more manegeable parts. For an LLM has no utility at all because if over-budget it can only use the response in the turn, if a manifest it has to be confident it will stay in context in the next call and has enough info to call the chunks. In a sense the chunks behave as "tools" but aren't part of the schema that is automatically injected in the context by the harness. And so they aren't as realiable.

So we'll leave the functins but its utitily  is contended. In summary, if the digest fit in context budget, no problem the user or the AI will use the context as intended (create a message, summarize, whatever). The tool call that returned the digest will be truncated and is what remains in the context. The model will see the (possibly) truncated message and it will be useful or not, it depends.

If too large, the model receives the manifest that becomes context automatically (but can be truncated) and can only request chunks in the subsequent tool use event loop. And what make with each chunk? It only can process them in the turn and genrate transient artifacts, it can't become context because the summ of all chunks if over-budget and becoming context would defeat the purpose of the digest in the first place.

So, digest and chunks are transient artifacts, the budget hint only serve to indicate user and AIs if =the digest can become context. If over-budget can be usable in the same turn of the tool call. If the model want to access the digest in future calls it should have to call it again., but in context should be enough info to recall how and for what call the digest tool.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — that framing is much clearer and I think it should guide the docs/API.

The key distinction is:

- **digest text is a transient artifact**
- **dialog messages/files are durable context**
- **tool responses are not reliable durable context**, especially with truncation

So budget handling should not pretend to solve context persistence. It only answers:

> Can this digest likely be placed into context as-is?

If yes, good — return it. The caller may paste it, link it, create a note, or use it immediately.

If no, returning a manifest is not a “continuation protocol” in any robust sense. It is just a deterministic diagnostic/index saying:

> This selection is too large at available disclosure levels. Here are possible parts if you want to inspect or materialize them.

That means we can keep chunking, but its role should be modest:

- useful for human inspection
- useful for creating multiple digest messages
- useful for debugging/budget diagnostics
- not a reliable autonomous LLM navigation scheme

I’d update `dialog_digest_docs()` to say this explicitly. Something like:

```python
def dialog_digest_docs() -> str:
    "Explain solveit dialog digest frontmatter, disclosure profiles, and usage."
    return """
# Solveit dialog digest docs

A dialog digest is a selected, disclosed, serialized view of a Solveit dialog for AI consumption.

## Important model

Digest strings and chunks are transient artifacts. They are useful in the current turn or when materialized into dialog messages/files. Tool responses may be truncated by the harness and should not be treated as durable context.

If a digest fits the requested budget, it can usually be used directly as context, pasted into a note, or written into a target dialog.

If a digest does not fit, the manifest is a compact diagnostic/index, not a reliable autonomous navigation protocol. Chunks can be requested for inspection or materialization, but repeatedly fetching chunks does not make the whole over-budget digest durable model context.

## Frontmatter fields

- `digest`: digest kind, currently `solveit-dialog`
- `path`: source dialog path
- `exported_at`: UTC timestamp
- `description`: human-readable purpose of this digest
- `profile`: named disclosure profile, if used
- `disclosure`: exact note/code/output disclosure settings
- `stats.total`: number of messages in the source selection
- `stats.exported`: number of serialized messages in the digest
- `conventions`: serialization conventions for the chosen format
- `docs`: helper function to call for this explanation

## Disclosure profiles

- `discovery`: section/map overview; note headers and code skeletons
- `compact`: compact narrative; note header plus first paragraph and code skeletons
- `handoff`: good continuation context; notes plus code signatures/docstrings
- `reuse`: fuller source for using functions elsewhere
- `archive`: full notes, full code, and outputs

## Important semantics

Disclosure is deterministic. Messages are not randomly summarized, truncated, or rewritten by an LLM.

Partial disclosure means the digest may show only a header, skeleton, or signature-level view of a message. The original dialog remains the source of truth.

`profile` names are convenience presets. The exact `disclosure` field is authoritative.

## Selection vs disclosure vs serialization

Selection chooses which messages are included.

Disclosure chooses how much of each selected message is shown.

Serialization chooses the output format: markdown, XML, or JSON.

## Execution correctness requirement

Disclosed code at every level should be valid Python that can be executed in the appropriate dialog/export context. Disclosure is not responsible for dependency isolation: callers must include the necessary import cells, private helpers, decorators, base classes, and other dependencies in the selected/exported message set. Full or partial isolation is handled separately by isolation tooling.

## Recommended AI behavior

Read the frontmatter first.

If `profile` is not `archive`, assume some content is intentionally disclosed rather than complete.

Use message ids to refer to exact source messages.

If the digest is too large and a manifest is returned, treat it as an index/diagnostic. Request chunks only for immediate inspection or to materialize them as messages/files.
""".strip()
```

And maybe a tiny note near the chunking helpers:

```markdown
Chunking is a deterministic inspection/materialization aid, not a robust mechanism for keeping over-budget context available to an LLM across turns.
```

That resolves the ambiguity without deleting the code.

<details class='token-usage-details' markdown='1'><summary>$0.0123</summary>

`total=78,470 | in=77,553 | out=917 | cached=2.5% | searches=0 | $0.0123`

</details>


## next

## branching
1. ephemeral -> git worktree?
2. permanent
How to do dialog branching:
- simple, no tracking: dup dialog -> create new dialog with no references in resulting dialogs.
- simple, with origin tracking: dup dialog -> the new dialog point to the origin. Fragile: origin can change and stale the chain.
- involved, git tracking: clean status (persistent commit before branching): branch stores the commit hash, we can reconstruct history or even attempt merging/syncing.

## dependency tracking\isolation
It is an important goal. It is in my list and I got first steps with my isolated dialog (prj/pote/explorer/isolated/isolated) and my dutil.flakes module. We'll tackle that later (probably in another session).

## import

Brings messages from other dialogs into the current one — selecting by symbol name, message id, or regex — and optionally run them on insertion.

see <a class="uk-link" href="/dialog_?name=prj%2Fdutil%2Fexplorer%2Fimportdlg%2Fimportdlg" target="_blank">importdlg exploratory dialog</a>

In [ ]:
#| export
def _r(m:dict, **kwargs):
    "Add kwargs to m and filter out key/value not compatible with add_msg or update_msg"
    if kwargs: m.update(kwargs)
    if m.msg_type == 'code': m.output = ''
    return dict(content=m.content, msg_type=m.msg_type, output=m.output, is_exported=m.is_exported, skipped=m.skipped, pinned=m.pinned, i_collapsed=m.i_collapsed, o_collapsed=m.o_collapsed, heading_collapsed=m.heading_collapsed)

In [ ]:
#| export
def find_symbol_msg(msgs, sym:str) -> dict|None:
    "Find last message in msgs list containing symbol definition (module level)"
    for m in reversed(msgs):
        if m.msg_type != 'code': continue
        root = ast_py(m.content)
        for node in root.find_all(pattern='def $FUNC($$$)'):
            if node.field('name').text() == sym: return m
        for node in root.find_all(pattern='class $CLASS'):
            if node.field('name').text() == sym: return m
        for node in root.find_all(pattern='$VAR = $$$'):
            if node.field('left').text() == sym: return m
    return None

In [ ]:
#| export
async def importdlg(
    dname:str='',  # dialog to import from; relative to solveit root (if starts with /) or current dialog; empty string for current dialog
    syms:list[str]=None,  # symbol names to import (finds last def/class/assignment); if provided, regex is ignored
    ids:list[str]=None,  # message ids to import; can combine with syms; if syms or ids provided, regex is ignored
    re_include:str=None,  # regex to filter messages (content+output); only used if no syms/ids; uses re.MULTILINE|re.DOTALL
    re_exclude:str=None,  # regex to exclude from re_include results; only used if no syms/ids
    run:bool=False,  # run imported messages after insertion
    header:bool=False,  # wrap imports in collapsed header with dialog name
    heading_collapsed:bool=False  # collapse the header after creation
):
    "Import messages from dname by symbols, ids, regex filter, or all messages; inserts below current message in original order"
    msgs, to_import = await find_msgs(dname=dname, include_meta=True, include_output=True), []
    if not dname: msgs = msgs[:await msg_idx()]
    if syms or ids:
        id2idx, seen = {m.id:i for i,m in enumerate(msgs)}, set()
        if syms:
            for sym in syms:
                if (m := find_symbol_msg(msgs, sym)) and m.id not in seen: to_import.append(m); seen.add(m.id)
        if ids:
            for m in msgs:
                if m.id in ids and m.id not in seen: to_import.append(m); seen.add(m.id)
        to_import.sort(key=lambda m: id2idx[m.id])
    else:
        to_import = msgs
        if re_include: to_import = [m for m in to_import if re.search(re_include, m.content + m.output, re.MULTILINE | re.DOTALL)]
        if re_exclude: to_import = [m for m in to_import if not re.search(re_exclude, m.content + m.output, re.MULTILINE | re.DOTALL)]
    new_ids = [await add_msg(**_r(m)) for m in to_import]
    if header and new_ids: 
        hdrid = await add_msg(f"## Imported from {dname}", placement='add_before', id=new_ids[0])
        new_ids.insert(0, hdrid)
        new_ids.append(await add_msg("## ----", id=new_ids[-1]))
    # time.sleep(0.5)
    if run and new_ids: await run_msg(','.join(new_ids))
    # time.sleep(0.1)
    if heading_collapsed and header: await toggle_header(hdrid)
    return new_ids

## export -

In [ ]:
from pote.dutil import ctxusage
await ctxusage()

91552

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose" markdown="1">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()